# 01 — Raw Collection | Full Scraping Bertahap (Production-Safe)

Penelitian sentimen temporal tentang **keputusan penetapan/tidak ditetapkannya status bencana nasional** di X/Twitter: Lombok 2018, Palu 2018, dan Sumatra 2025.

**KONDISI AWAL: PREVIEW-ONLY.** Tidak ada panggilan berbayar saat menjalankan semua cell dengan konfigurasi bawaan. Jangan mengaktifkan `EXECUTE_SCRAPING` sebelum mengonfirmasi saldo tersedia, memahami risiko biaya, dan melihat preview serta hasil tes.

Versi ini mempertahankan `CONFIG_VERSION`, `ACTOR_ID`, `task_key()`, struktur folder dan checkpoint dari pipeline pilot. Hasil pilot tetap dibaca dan tidak dihapus. **Simpan notebook di `01_Raw Data/namafile.ipynb`

**Prinsip:** `E01_P01` satu query awal per kasus, jendela 6 jam, round-robin, hanya sedikit Actor calls per batch, batas USD per run dan per sesi, checkpoint append-only, serta penghitungan ID unik global. `MAX_ITEMS` input Actor bukan bukti kelengkapan.

> Catatan anggaran: pengguna melaporkan **$1,21 sudah digunakan**. Itu bukan saldo tersisa. Nilai `AVAILABLE_CREDIT_USD` harus berasal dari dashboard Apify **saat batch akan dijalankan**; tidak boleh dihitung dengan mengurangkan asumsi tertentu.

**Pembaharuan v2:** prioritas fase keputusan tanpa menghapus tanggal, pause 3 run berturut-turut tanpa tweet asli, status data terpisah dari SAVED, rekonsiliasi usage $1.21→$1.22, dan laporan global lintas pilot/full.


In [37]:
# 01 — IMPORT & VERSI
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import time
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from decimal import Decimal, InvalidOperation
from getpass import getpass
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import pandas as pd
from IPython.display import display

try:
    from apify_client import ApifyClient
except ImportError as exc:
    raise ImportError(
        "Instal apify-client dan pandas pada kernel VS Code yang dipakai. "
        "Rekomendasi: uv pip install apify-client pandas"
    ) from exc

print('Pandas:', pd.__version__)
print('Import berhasil (belum ada panggilan API).')


Pandas: 3.0.6
Import berhasil (belum ada panggilan API).


In [38]:
# 02 — LOKASI PROJECT & OUTPUT
# Repository memiliki folder bernama "01_Raw Data" (spasi), bukan "01_Raw_Data".
# Jika Jupyter dibuka dari lokasi lain, set PROJECT_ROOT melalui environment variable.
root_env = os.getenv('PROJECT_ROOT')
if root_env:
    PROJECT_ROOT = Path(root_env).expanduser().resolve()
elif (Path.cwd() / '01_Raw Data').is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
elif Path.cwd().name == '01_Raw Data':
    PROJECT_ROOT = Path.cwd().parent.resolve()
else:
    raise RuntimeError(
        'Folder 01_Raw Data tidak ditemukan. Buka notebook dari repository atau set PROJECT_ROOT.'
    )

STAGE_DIR = PROJECT_ROOT / '01_Raw Data'
OUT_DIR = STAGE_DIR / 'outputs'
RAW_DIR = OUT_DIR / 'raw_jsonl'
CSV_DIR = OUT_DIR / 'raw_csv'
AUDIT_DIR = OUT_DIR / 'audit'
STATE_DIR = OUT_DIR / 'state'
REPORT_DIR = OUT_DIR / 'reports'
for folder in [RAW_DIR, CSV_DIR, AUDIT_DIR, STATE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = STATE_DIR / 'collection_checkpoint.jsonl'
CONFIG_VERSION = 'policy_v2_2026_10'
ACTOR_ID = 'CJdippxWmn9uRfooo'
WIB = ZoneInfo('Asia/Jakarta')
UTC = timezone.utc

print('Project:', PROJECT_ROOT)
print('Output :', OUT_DIR)
print('Actor  :', ACTOR_ID)

Project: C:\Penelitian Sentimen Bencana Nasional
Output : C:\Penelitian Sentimen Bencana Nasional\01_Raw Data\outputs
Actor  : CJdippxWmn9uRfooo


## Langkah 1 — Konfigurasi kasus, kata kunci, dan query
Ketiga periode **inklusif**. Semua tanggal tweet dan jendela pengambilan dihitung dalam WIB.
Kelompok EVENT harus berpasangan dengan kelompok POLICY. Tidak ada scraping event-only.
Keyword luas (contohnya "status darurat", "pemerintah pusat") ditandai untuk peninjauan,
bukan otomatis dianggap sebagai opini tentang status bencana nasional.

In [39]:
# 03 — KONFIGURASI PENELITIAN (dipertahankan dari notebook semula)
DISASTER_CASES = {
    'lombok_2018': {
        'label': 'Gempa Lombok 2018',
        'start_date': '2018-07-29', 'end_date': '2018-08-31',
        'pilot_date': '2018-08-06',
        'event_terms': [
            'Lombok', 'gempa Lombok', 'gempa bumi Lombok', 'gempa NTB',
            'gempa bumi NTB', 'NTB', 'Nusa Tenggara Barat', 'Mataram',
            'Lombok Utara', 'Lombok Barat', 'Lombok Timur', 'Lombok Tengah'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'jadiin bencana nasional'
        ],
    },
    'palu_2018': {
        'label': 'Gempa dan Tsunami Palu 2018',
        'start_date': '2018-09-28', 'end_date': '2018-10-31',
        'pilot_date': '2018-09-29',
        'event_terms': [
            'Palu', 'gempa Palu', 'gempa bumi Palu', 'tsunami Palu',
            'gempa tsunami Palu', 'Donggala', 'gempa Donggala',
            'tsunami Donggala', 'Sulawesi Tengah', 'gempa Sulawesi Tengah',
            'tsunami Sulawesi Tengah', 'bencana Palu', 'bencana Sulawesi Tengah'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'bantuan internasional'
        ],
    },
    'sumatra_2025': {
        'label': 'Banjir dan Longsor Sumatra 2025',
        'start_date': '2025-11-26', 'end_date': '2025-12-31',
        'pilot_date': '2025-11-29',
        'event_terms': [
            'Aceh', 'banjir Aceh', 'longsor Aceh', 'bencana Aceh',
            'Sumatra Utara', 'Sumatera Utara', 'banjir Sumatra Utara',
            'banjir Sumatera Utara', 'longsor Sumatra Utara',
            'longsor Sumatera Utara', 'bencana Sumatra Utara',
            'bencana Sumatera Utara', 'Sumatra Barat', 'Sumatera Barat',
            'banjir Sumatra Barat', 'banjir Sumatera Barat',
            'longsor Sumatra Barat', 'longsor Sumatera Barat',
            'bencana Sumatra Barat', 'bencana Sumatera Barat',
            'banjir Sumatra', 'banjir Sumatera', 'longsor Sumatra',
            'longsor Sumatera', 'bencana Sumatra', 'bencana Sumatera'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'prioritas nasional',
            'penanganan nasional', 'penanganan berskala nasional',
            'berskala nasional', 'ditangani pemerintah pusat',
            'penanganan pemerintah pusat', 'pemerintah pusat',
            'status daerah', 'status provinsi'
        ],
    },
}

STRONG_POLICY_TERMS = [
    'bencana nasional', 'status bencana nasional',
    'penetapan bencana nasional', 'ditetapkan sebagai bencana nasional',
    'ditetapkan bencana nasional', 'tetapkan bencana nasional',
    'menetapkan bencana nasional', 'jadi bencana nasional',
    'jadiin bencana nasional', 'status nasional'
]

BROAD_POLICY_TERMS = [
    'penetapan status bencana', 'penetapan status', 'status bencana',
    'status darurat', 'bencana daerah', 'status bencana daerah',
    'bantuan internasional', 'prioritas nasional', 'penanganan nasional',
    'penanganan berskala nasional', 'berskala nasional',
    'ditangani pemerintah pusat', 'penanganan pemerintah pusat',
    'pemerintah pusat', 'status daerah', 'status provinsi'
]

# Grup query yang sama seperti notebook lama (5 event × 6 policy / batch).
EVENT_CHUNK_SIZE = 5
POLICY_CHUNK_SIZE = 6


def chunks(items: list[str], size: int):
    if size <= 0:
        raise ValueError('Ukuran batch harus > 0')
    for i in range(0, len(items), size):
        yield items[i:i + size]


def or_query(terms: list[str]) -> str:
    return '(' + ' OR '.join('"' + term.replace('"', '') + '"' for term in terms) + ')'


def make_query_batches(case_config: dict) -> list[dict]:
    batches = []
    for e_index, events in enumerate(chunks(case_config['event_terms'], EVENT_CHUNK_SIZE), 1):
        for p_index, policies in enumerate(chunks(case_config['policy_terms'], POLICY_CHUNK_SIZE), 1):
            # X search: spasi antarkelompok bermakna konjungsi/AND.
            # Keyword kebijakan tetap diwajibkan; hindari literal 'AND' dalam kueri.
            base = f'{or_query(events)} {or_query(policies)}'
            batches.append({
                'query_id': f'E{e_index:02d}_P{p_index:02d}',
                'base_query': base,
                'event_terms': events,
                'policy_terms': policies,
            })
    return batches

for case_id, conf in DISASTER_CASES.items():
    conf['query_batches'] = make_query_batches(conf)
    print(case_id, '| event:', len(conf['event_terms']),
          '| policy:', len(conf['policy_terms']),
          '| query batch:', len(conf['query_batches']))

assert [len(DISASTER_CASES[c]['query_batches']) for c in DISASTER_CASES] == [9, 9, 24]

lombok_2018 | event: 12 | policy: 16 | query batch: 9
palu_2018 | event: 13 | policy: 16 | query batch: 9
sumatra_2025 | event: 26 | policy: 24 | query batch: 24


## Langkah 2 — Pilih strategi produksi tanpa API call

- **Selalu mulai dengan `EXECUTE_SCRAPING=False`**; `RUN_MODE="full"` saja tidak memanggil API.
- **Jangan mengganti `CONFIG_VERSION`** kecuali memang ingin membuat checkpoint baru untuk query berbeda. Format `task_key` dipertahankan agar task pilot 6 jam dengan query sama tidak terbayar ulang.
- `FULL_WINDOW_HOURS=6` dan `FULL_QUERY_IDS` hanya memilih `E01_P01`. Ini **belum** mencakup seluruh query atau seluruh opini; query alternatif dapat ditambahkan pada fase berikutnya setelah melihat yield dan cakupan temporal.
- `FULL_HOUR_PRIORITY=(6, 12, 18, 0)` memprioritaskan jam yang pada sampel pilot lebih produktif; **seluruh empat jendela harian tetap direncanakan**, sehingga dataset akhir tidak boleh dianalisis sebelum cakupan hari lengkap.
- Target **4.000–5.000 ID tweet asli unik *relevan*** adalah sasaran, bukan jaminan. Pemberitaan, promosi, dan tweet di luar kebijakan perlu divalidasi semantik.
- `max_items` dan `max_total_charge_usd` adalah **parameter tingkat run pada Apify API**, bukan sekadar properti input Actor. Tetap cek harga Actor serta biaya aktual dashboard.


In [75]:
# 04XRun — KONFIGURASI FULL COLLECTION BERTAHAP (DEFAULT: TIDAK BERBAYAR)
RUN_MODE = 'full'
EXECUTE_SCRAPING = False
PAID_RUN_CONFIRMATION = ''  # Hanya isi jika memang siap: I_APPROVE_PAID_FULL_SCRAPING

# Pembaruan dashboard sesudah smoke test: $1.22 terpakai dari batas $5.
CREDIT_USED_TO_DATE_USD = Decimal('1.40')  # Informasi; tidak dipakai menghitung sisa saldo.
AVAILABLE_CREDIT_USD = Decimal('3.60')               # WAJIB isi sesuai sisa kredit dashboard Apify sebelum PAID.
CREDIT_RESERVE_USD = Decimal('0.50')       # Jangan gunakan cadangan ini.
MAX_SESSION_SPEND_USD = Decimal('0.03')    # Batas komitmen maksimal SATU sesi notebook.
MAX_CHARGE_PER_RUN_USD = Decimal('0.03')   # Apify max_total_charge_usd per run.
MAX_NEW_RUNS = 1                           # Batch kecil, maksimal 3 panggilan baru.
MAX_CONSECUTIVE_NO_REAL = 3                # Pause sesi setelah 3 run baru berturut-turut 0 tweet asli.
MAX_ITEMS = 20                             # Hint Actor + server cap max_items (bukan batas raw pasti).
RETRY_UNCERTAIN_TASKS = False              # Jangan bayar ulang task berstatus NEEDS_REVIEW.

# Pengumpulan lintas kasus: satu query pertama per kasus, seluruh tanggal, jendela 6 jam.
FULL_CASES = ('lombok_2018', 'palu_2018', 'sumatra_2025')
FULL_QUERY_IDS = {
    'lombok_2018': ('E01_P01',),
    'palu_2018': ('E01_P01',),
    'sumatra_2025': ('E01_P01',),
}
FULL_WINDOW_HOURS = 6
FULL_HOUR_PRIORITY = (6, 12, 18, 0)  # Semua jam masih dikunjungi; prioritas saja.
# Tanggal berpotensi ramai DIDAHULUKAN saja; tanggal lain tetap masuk rencana FULL.
FULL_PRIORITY_DATES = {
    'lombok_2018': ('2018-08-14', '2018-08-13'),
    'palu_2018': ('2018-10-03', '2018-10-02'),
    'sumatra_2025': ('2025-12-04', '2025-12-03'),
}
FULL_MAX_DAYS_PER_CASE = None  # Mis. 2 untuk uji rencana, bukan rekomendasi dataset final.

# Tetap menyediakan mode pilot untuk regresi logika; full mode yang digunakan saat produksi.
PILOT_QUERY_LIMIT = 1
PILOT_HOURS = 6
PILOT_START_HOUR = 6
ENABLE_ADAPTIVE = False                 # Default OFF karena dapat membuat panggilan tambahan.
MAX_ADAPTIVE_DEPTH = 3
REQUEST_DELAY_SECONDS = 1.0

assert RUN_MODE in {'pilot', 'full'}
assert 1 <= PILOT_HOURS <= 24 and 0 <= PILOT_START_HOUR <= 23
assert PILOT_START_HOUR + PILOT_HOURS <= 24
assert FULL_WINDOW_HOURS in {1, 3, 6, 12, 24}
assert MAX_NEW_RUNS >= 1 and MAX_ITEMS >= 1
assert MAX_CONSECUTIVE_NO_REAL >= 2
for _c, _days in FULL_PRIORITY_DATES.items():
    assert _c in DISASTER_CASES and len(set(_days)) == len(_days)
    assert all(DISASTER_CASES[_c]['start_date'] <= d <= DISASTER_CASES[_c]['end_date'] for d in _days)
assert MAX_SESSION_SPEND_USD > 0 and MAX_CHARGE_PER_RUN_USD > 0
assert MAX_SESSION_SPEND_USD >= MAX_CHARGE_PER_RUN_USD
assert set(FULL_CASES).issubset(DISASTER_CASES)
assert len(FULL_HOUR_PRIORITY) == 24 // FULL_WINDOW_HOURS
assert sorted(FULL_HOUR_PRIORITY) == list(range(0, 24, FULL_WINDOW_HOURS))
assert set(FULL_QUERY_IDS) == set(FULL_CASES)
assert not RETRY_UNCERTAIN_TASKS, 'Retry task ambigu harus dilakukan manual, bukan otomatis.'

for _case_id in FULL_CASES:
    _available_query_ids = {b['query_id'] for b in DISASTER_CASES[_case_id]['query_batches']}
    assert set(FULL_QUERY_IDS[_case_id]).issubset(_available_query_ids)


def wib_dt(day: str, hour: int = 0) -> datetime:
    return datetime.fromisoformat(day).replace(hour=hour, tzinfo=WIB)


def iso_wib(dt: datetime) -> str:
    return dt.astimezone(WIB).strftime('%Y-%m-%d %H:%M:%S%z')


def make_search_query(base_query: str, start_wib: datetime, end_wib: datetime) -> str:
    start_unix = int(start_wib.timestamp())
    end_unix = int(end_wib.timestamp())
    return f'{base_query} lang:in since_time:{start_unix} until_time:{end_unix}'


# IDENTITAS TASK DARI PILOT DIPERTAHANKAN — JANGAN UBAH TANPA ALASAN.
def task_key(task: dict) -> str:
    identifying = {
        'case': task['case'], 'base_query': task['base_query'],
        'start_wib': iso_wib(task['start_wib']),
        'end_wib': iso_wib(task['end_wib']),
        'config_version': CONFIG_VERSION, 'actor_id': ACTOR_ID,
    }
    digest = hashlib.sha256(
        json.dumps(identifying, sort_keys=True, ensure_ascii=False).encode('utf-8')
    ).hexdigest()[:20]
    return f"{task['case']}_{digest}"


def task_info(case_id: str, batch: dict, start: datetime, end: datetime, depth=0) -> dict:
    task = {
        'case': case_id, 'query_id': batch['query_id'],
        'base_query': batch['base_query'],
        'start_wib': start, 'end_wib': end, 'depth': depth,
    }
    task['task_id'] = task_key(task)
    task['query'] = make_search_query(task['base_query'], start, end)
    return task


def date_windows(conf: dict, hours: int):
    start = wib_dt(conf['start_date'])
    end_exclusive = wib_dt(conf['end_date']) + timedelta(days=1)
    step = timedelta(hours=hours)
    while start < end_exclusive:
        stop = min(start + step, end_exclusive)
        yield start, stop
        start = stop


def build_tasks(mode: str = RUN_MODE) -> list[dict]:
    # Round-robin antar kasus dan hari. Mode full tidak memproses 42 batch query.
    case_tasks = {}
    for case_id, conf in DISASTER_CASES.items():
        group = []
        if mode == 'pilot':
            start = wib_dt(conf['pilot_date'], PILOT_START_HOUR)
            windows = [(start, start + timedelta(hours=PILOT_HOURS))]
            batches = conf['query_batches'][:PILOT_QUERY_LIMIT]
        elif mode == 'full':
            if case_id not in FULL_CASES:
                case_tasks[case_id] = []
                continue
            grouped = defaultdict(list)
            for start, end in date_windows(conf, FULL_WINDOW_HOURS):
                grouped[start.date()].append((start, end))
            dates = sorted(grouped.keys())
            # Prioritas tanggal mengubah urutan, BUKAN membuang hari.
            preferred = [datetime.fromisoformat(d).date()
                        for d in FULL_PRIORITY_DATES.get(case_id, ())]
            dates = [d for d in preferred if d in grouped] + [
                d for d in dates if d not in preferred]
            if FULL_MAX_DAYS_PER_CASE is not None:
                dates = dates[:FULL_MAX_DAYS_PER_CASE]
            windows = []
            for date in dates:
                per_day = sorted(
                    grouped[date],
                    key=lambda pair: FULL_HOUR_PRIORITY.index(pair[0].hour),
                )
                windows.extend(per_day)
            batch_by_id = {b['query_id']: b for b in conf['query_batches']}
            batches = [batch_by_id[qid] for qid in FULL_QUERY_IDS[case_id]]
        else:
            raise ValueError('mode harus pilot atau full')
        for start, end in windows:
            for batch in batches:
                group.append(task_info(case_id, batch, start, end))
        case_tasks[case_id] = group

    tasks = []
    longest = max((len(group) for group in case_tasks.values()), default=0)
    for index in range(longest):
        for case_id in DISASTER_CASES:
            if index < len(case_tasks[case_id]):
                tasks.append(case_tasks[case_id][index])
    return tasks


def preview_plan(mode=RUN_MODE):
    tasks = build_tasks(mode)
    print('=' * 68)
    print('PREVIEW-ONLY | mode:', mode, '| tugas awal:', len(tasks))
    print('Full window:', FULL_WINDOW_HOURS, 'jam | batas panggilan per sesi:', MAX_NEW_RUNS)
    print('EXECUTE_SCRAPING:', EXECUTE_SCRAPING,
        '| PAID_RUN_CONFIRMATION:', 'UNLOCKED' if PAID_RUN_CONFIRMATION else 'LOCKED')
    print('Saldo terpakai (informasi): $', CREDIT_USED_TO_DATE_USD,
        '| saldo TERSEDIA:', AVAILABLE_CREDIT_USD)
    print('Batas sesi: $', MAX_SESSION_SPEND_USD,
        '| per run: $', MAX_CHARGE_PER_RUN_USD)
    for case in DISASTER_CASES:
        print(' ', case, ':', sum(t['case'] == case for t in tasks), 'tasks')
    for task in tasks[:min(6, len(tasks))]:
        print(' -', task['case'], task['query_id'],
            iso_wib(task['start_wib']), '→', iso_wib(task['end_wib']))
    return tasks

TASK_PREVIEW = preview_plan()


PREVIEW-ONLY | mode: full | tugas awal: 416
Full window: 6 jam | batas panggilan per sesi: 1
EXECUTE_SCRAPING: False | PAID_RUN_CONFIRMATION: LOCKED
Saldo terpakai (informasi): $ 1.40 | saldo TERSEDIA: 3.60
Batas sesi: $ 0.03 | per run: $ 0.03
  lombok_2018 : 136 tasks
  palu_2018 : 136 tasks
  sumatra_2025 : 144 tasks
 - lombok_2018 E01_P01 2018-08-14 06:00:00+0700 → 2018-08-14 12:00:00+0700
 - palu_2018 E01_P01 2018-10-03 06:00:00+0700 → 2018-10-03 12:00:00+0700
 - sumatra_2025 E01_P01 2025-12-04 06:00:00+0700 → 2025-12-04 12:00:00+0700
 - lombok_2018 E01_P01 2018-08-14 12:00:00+0700 → 2018-08-14 18:00:00+0700
 - palu_2018 E01_P01 2018-10-03 12:00:00+0700 → 2018-10-03 18:00:00+0700
 - sumatra_2025 E01_P01 2025-12-04 12:00:00+0700 → 2025-12-04 18:00:00+0700


In [41]:
# 04A — PREVIEW QUERY ALTERNATIF TANPA MEMANGGIL APIFY
# Belum mengaktifkan query tambahan. Setelah hasil E01_P01 dievaluasi,
# pertimbangkan E02_P01 / E03_P01 untuk variasi lokasi.
ALTERNATIVE_QUERY_IDS = {
    'lombok_2018': ('E02_P01',),
    'palu_2018': ('E02_P01',),
    'sumatra_2025': ('E02_P01', 'E03_P01', 'E04_P01'),
}

def preview_alternative_queries():
    for case, ids in ALTERNATIVE_QUERY_IDS.items():
        indexed = {b['query_id']: b for b in DISASTER_CASES[case]['query_batches']}
        print('\n', case)
        for qid in ids:
            b = indexed[qid]
            print(' ', qid, '| EVENT:', ', '.join(b['event_terms']))
            print(' ', ' | POLICY:', ', '.join(b['policy_terms']))
    print('Tidak ada query alternatif yang dijalankan; FULL_QUERY_IDS tetap E01_P01.')

preview_alternative_queries()



 lombok_2018
  E02_P01 | EVENT: NTB, Nusa Tenggara Barat, Mataram, Lombok Utara, Lombok Barat
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional, penetapan status bencana, penetapan status

 palu_2018
  E02_P01 | EVENT: Donggala, gempa Donggala, tsunami Donggala, Sulawesi Tengah, gempa Sulawesi Tengah
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional, penetapan status bencana, penetapan status

 sumatra_2025
  E02_P01 | EVENT: Sumatera Utara, banjir Sumatra Utara, banjir Sumatera Utara, longsor Sumatra Utara, longsor Sumatera Utara
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional, penetapan status bencana, penetapan status
  E03_P01 | EVENT: bencana Sumatra Utara, bencana Sumatera Utara, Sumatra Barat, Sumatera Barat, banjir Sumatra Barat
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional,

In [42]:
# 05 — VALIDASI PRE-FLIGHT TANPA API

def preflight(tasks: list[dict], mode=RUN_MODE):
    assert len(DISASTER_CASES) == 3
    assert {c: len(v['query_batches']) for c, v in DISASTER_CASES.items()} == {
        'lombok_2018': 9, 'palu_2018': 9, 'sumatra_2025': 24,
    }
    assert MAX_NEW_RUNS > 0
    assert len({t['task_id'] for t in tasks}) == len(tasks), 'Task ID duplikat'
    for task in tasks:
        assert task['start_wib'] < task['end_wib']
        assert 'lang:in' in task['query']
        assert 'since_time:' in task['query'] and 'until_time:' in task['query']
        assert ' OR ' in task['base_query']
        assert ') (' in task['base_query'], 'Query harus EVENT dan POLICY'
        assert ' AND ' not in task['base_query'], 'Hapus literal AND dari query'
        assert task['case'] in DISASTER_CASES
        if mode == 'full':
            assert task['case'] in FULL_CASES
            assert task['query_id'] in FULL_QUERY_IDS[task['case']]
            assert (task['end_wib'] - task['start_wib']) == timedelta(hours=FULL_WINDOW_HOURS)
    return True

assert preflight(TASK_PREVIEW)
print('PREFLIGHT OFFLINE: PASS')


PREFLIGHT OFFLINE: PASS


## Langkah 3 — Validasi Actor, format tweet, dan pembagian jendela
Yang divalidasi: tweet asli (`type=tweet`), ID, `createdAt` parseable, tanggal dalam jendela,
serta dua kelompok keyword. Hasil yang meragukan tetap disimpan ke arsip raw dan audit.
Catatan: pencocokan teks adalah indikator otomatis, bukan keputusan relevansi semantik final.

In [43]:
# 06 — FUNGSI AUDIT DATA (TANPA API CALL)

def match_any_phrase(text: str, terms: list[str]) -> bool:
    # Hindari pencocokan sebagian kata (misalnya 'Palu' di 'kepaluan').
    sample = str(text).casefold()
    for term in terms:
        phrase = term.casefold()
        pat = r'(?<!\w)' + re.escape(phrase) + r'(?!\w)'
        if re.search(pat, sample):
            return True
    return False

# ALIAS LOKASI KHUSUS AUDIT
# Tidak mengubah query scraping atau task_key
AUDIT_EVENT_ALIASES = {
    'palu_2018': [
        'Sulteng',
    ],
}


def get_audit_event_terms(case_id: str) -> list[str]:
    original_terms = DISASTER_CASES[case_id]['event_terms']
    aliases = AUDIT_EVENT_ALIASES.get(case_id, [])

    return original_terms + aliases

def parse_twitter_timestamp(values: pd.Series) -> pd.Series:
    result = pd.to_datetime(
        values, format='%a %b %d %H:%M:%S %z %Y',
        errors='coerce', utc=True
    )
    # Cadangan apabila format sumber suatu hari berubah.
    missing = result.isna() & values.notna()
    if missing.any():
        result.loc[missing] = pd.to_datetime(
            values.loc[missing], errors='coerce', utc=True
        )
    return result


def audit_items(items: list[dict], task: dict) -> tuple[pd.DataFrame, dict]:
    df = pd.DataFrame(items)
    initial_columns = list(df.columns)
    for column in ['type', 'id', 'createdAt', 'text', 'lang']:
        if column not in df.columns:
            df[column] = pd.Series([None] * len(df), dtype='object')

    timestamps = parse_twitter_timestamp(df['createdAt'])
    start_utc = pd.Timestamp(task['start_wib'].astimezone(UTC))
    end_utc = pd.Timestamp(task['end_wib'].astimezone(UTC))

    df['audit_created_at_utc'] = timestamps
    df['audit_is_real_tweet'] = df['type'].eq('tweet')
    df['audit_in_window'] = timestamps.ge(start_utc) & timestamps.lt(end_utc)

    # Audit menggunakan seluruh daftar istilah yang relevan dengan kasus.
    # Term yang ada dalam batch query sendiri juga wajib dicek melalui hasil.
    event_terms = get_audit_event_terms(task['case'])
    policy_terms = DISASTER_CASES[task['case']]['policy_terms']
    texts = df['text'].fillna('').astype(str)
    df['audit_has_event'] = texts.map(lambda x: match_any_phrase(x, event_terms))
    df['audit_has_policy'] = texts.map(lambda x: match_any_phrase(x, policy_terms))
    df['audit_policy_strong'] = texts.map(lambda x: match_any_phrase(x, STRONG_POLICY_TERMS))
    df['audit_candidate'] = (
        df['audit_is_real_tweet'] & df['audit_in_window']
        & df['audit_has_event'] & df['audit_has_policy']
    )

    valid_ids = df.loc[df['audit_is_real_tweet'], 'id'].dropna().astype(str)
    n_mock = int(df['type'].eq('mock_tweet').sum())
    n_real = int(df['audit_is_real_tweet'].sum())
    n_in = int((df['audit_is_real_tweet'] & df['audit_in_window']).sum())
    n_candidate = int(df['audit_candidate'].sum())

    metrics = {
        'rows_raw': len(df),
        'rows_real_tweet': n_real,
        'rows_mock': n_mock,
        'rows_unknown_type': int((~df['type'].isin(['tweet', 'mock_tweet'])).sum()),
        'rows_real_in_window': n_in,
        'rows_real_out_window': int((df['audit_is_real_tweet'] & ~df['audit_in_window']).sum()),
        'rows_real_missing_timestamp': int((df['audit_is_real_tweet'] & timestamps.isna()).sum()),
        'rows_event_match': int((df['audit_is_real_tweet'] & df['audit_has_event']).sum()),
        'rows_policy_match': int((df['audit_is_real_tweet'] & df['audit_has_policy']).sum()),
        'rows_event_and_policy_in_window': n_candidate,
        'rows_strong_policy_candidate': int((df['audit_candidate'] & df['audit_policy_strong']).sum()),
        'unique_real_ids': int(valid_ids.nunique()),
        'unique_real_in_window_ids': int(df.loc[
            df['audit_is_real_tweet'] & df['audit_in_window'], 'id'
        ].dropna().astype(str).nunique()),
        'unique_policy_candidate_ids': int(df.loc[
            df['audit_candidate'], 'id'
        ].dropna().astype(str).nunique()),
        'duplicate_real_ids_in_dataset': int(valid_ids.duplicated().sum()),
        'raw_columns': initial_columns,
        'created_min_utc': timestamps.min().isoformat() if timestamps.notna().any() else None,
        'created_max_utc': timestamps.max().isoformat() if timestamps.notna().any() else None,
    }
    return df, metrics


def quality_flags(metrics: dict) -> list[str]:
    flags = []
    if metrics['rows_raw'] == 0:
        flags.append('EMPTY_SEARCH')
    if metrics['rows_mock']:
        flags.append('MOCK_PRESENT')
    if metrics['rows_raw'] > 0 and metrics['rows_real_tweet'] == 0:
        flags.append('NO_REAL_TWEET')
    if metrics['rows_real_out_window']:
        flags.append('OUT_OF_WINDOW')
    if metrics['rows_real_missing_timestamp']:
        flags.append('BAD_TIMESTAMP')
    if metrics['rows_real_tweet'] and metrics['rows_event_and_policy_in_window'] == 0:
        flags.append('NO_POLICY_CANDIDATE')
    # MAX_ITEMS bukan hard-limit aktual Actor. Ini indikator risiko saja.
    if metrics['unique_real_ids'] >= MAX_ITEMS:
        flags.append('POSSIBLE_TRUNCATION')
    return flags or ['BASIC_CHECKS_PASS']


# Tes cepat audit tanpa Apify dengan tanggal dan kasus otomatis.
_t = TASK_PREVIEW[0]
_event_term = DISASTER_CASES[_t['case']]['event_terms'][0]
_timedemo = (_t['start_wib'] + timedelta(minutes=2)).astimezone(UTC)
_sample = [
    {'type': 'mock_tweet', 'id': '-1', 'text': 'Actor mock placeholder'},
    {'type': 'tweet', 'id': 'demo-real',
    'text': f'Apakah {_event_term} perlu ditetapkan sebagai bencana nasional?',
    'createdAt': _timedemo.strftime('%a %b %d %H:%M:%S %z %Y'),
    'lang': 'id'},
]
_audit_demo, _m_demo = audit_items(_sample, _t)
assert _m_demo['rows_mock'] == 1 and _m_demo['rows_event_and_policy_in_window'] == 1
print('UNIT TEST AUDIT: PASS (tidak memanggil Apify)')

# UNIT TEST ALIAS LOKASI PALU
palu_terms = get_audit_event_terms('palu_2018')

assert match_any_phrase(
    'Gempa bumi dan tsunami Sulteng perlu menjadi bencana nasional',
    palu_terms
)

assert match_any_phrase(
    'Pemerintah menangani bencana di Sulawesi Tengah',
    palu_terms
)

assert not match_any_phrase(
    'Status bencana nasional perlu dipertimbangkan',
    palu_terms
)

print('UNIT TEST ALIAS PALU: PASS')


UNIT TEST AUDIT: PASS (tidak memanggil Apify)
UNIT TEST ALIAS PALU: PASS


In [44]:
# 06B — AUDIT RELEVANSI PERISTIWA (OFFLINE)
# Tidak mengubah DISASTER_CASES, query, task_id, atau checkpoint.

EVENT_SCOPE_AUDIT_VERSION = "event_scope_v1"

HISTORICAL_MARKERS = [
    "tsunami Aceh",
    "tsunami 2004",
    "Aceh 2004",
    "Sumatra-Andaman",
    "Sumatera-Andaman",
]

SUMATRA_SCOPE_LOCATIONS = [
    "Aceh", "Sumatra", "Sumatera",
    "Sumut", "Sumbar",
    "Aceh Utara", "Aceh Tamiang",
    "Tapanuli", "Agam",
]

SUMATRA_2025_DISASTER_TERMS = [
    "banjir", "banjir bandang", "longsor",
    "bencana ekologis", "bencana ekologi",
]

CURRENT_DECISION_CUES = [
    "belum", "tak kunjung", "sampai sekarang",
    "sekarang", "tak juga", "tidak juga",
    "tidak mau", "mendesak", "didesak",
    "desak", "minta", "meminta",
    "butuh", "perlu", "segera",
    "mengapa", "kenapa",
]

EXPLICIT_PRESENT_CUES = [
    "sampai sekarang", "sekarang", "saat ini",
    "hingga kini", "tak kunjung",
    "belum ditetapkan", "tahun ini",
]


def classify_event_scope(text: str, case_id: str):
    """
    Return (status_relevansi, alasan).
    Aturan heuristik konservatif, bukan label final.
    """
    text = str(text or "")

    historical = (
        bool(re.search(r"(?<!\w)2004(?!\w)", text))
        or match_any_phrase(text, HISTORICAL_MARKERS)
    )

    policy_match = match_any_phrase(
        text,
        DISASTER_CASES[case_id]["policy_terms"]
    )

    # Kasus 2018: gunakan lokasi peristiwa yang sudah ada.
    if case_id in ("lombok_2018", "palu_2018"):
        event_match = match_any_phrase(
            text, get_audit_event_terms(case_id)
        )

        if event_match and policy_match:
            if historical:
                return "TARGET_WITH_HISTORY", "case_and_history"
            return "TARGET_CURRENT", "case_and_policy"

        return "NEEDS_REVIEW", "insufficient_case_context"

    # Kasus banjir/longsor Sumatra 2025
    location = match_any_phrase(
        text, SUMATRA_SCOPE_LOCATIONS
    )

    hazard = match_any_phrase(
        text, SUMATRA_2025_DISASTER_TERMS
    )

    year_2025 = bool(
        re.search(r"(?<!\w)2025(?!\w)", text)
    )

    decision_now = (
        policy_match
        and match_any_phrase(text, CURRENT_DECISION_CUES)
    )

    strong_current = (
        location and policy_match
        and (hazard or year_2025)
    )

    weak_current = location and decision_now

    if historical:
        if strong_current:
            return (
                "TARGET_WITH_HISTORY",
                "current_disaster_and_history"
            )

        if (
            weak_current
            and match_any_phrase(text, EXPLICIT_PRESENT_CUES)
        ):
            return (
                "NEEDS_REVIEW",
                "history_with_ambiguous_current_reference"
            )

        return (
            "HISTORICAL_ONLY",
            "historical_without_2025_evidence"
        )

    if strong_current or weak_current:
        return (
            "TARGET_CURRENT",
            "current_event_or_decision_evidence"
        )

    return (
        "NEEDS_REVIEW",
        "insufficient_2025_event_evidence"
    )



# REGRESSION TEST — TANPA APIFY

test_cases = [
    (
        "Tsunami Aceh 2004 ditetapkan sebagai bencana nasional.",
        "HISTORICAL_ONLY",
    ),
    (
        "Banjir Aceh 2025 harus ditetapkan bencana nasional.",
        "TARGET_CURRENT",
    ),
    (
        "Tsunami Aceh 2004 ditetapkan sebagai bencana nasional, "
        "mengapa banjir Aceh 2025 belum?",
        "TARGET_WITH_HISTORY",
    ),
    (
        "Aceh 2004 dulu bencana nasional, "
        "sekarang mengapa status itu masih diperdebatkan?",
        "NEEDS_REVIEW",
    ),
]

for sample_text, expected in test_cases:
    result, reason = classify_event_scope(
        sample_text, "sumatra_2025"
    )

    assert result == expected, (
        f"Expected {expected}, got {result}: {sample_text}"
    )

    print(f"PASS: {result} | {reason}")

print("\nEVENT SCOPE UNIT TEST: PASS")


PASS: HISTORICAL_ONLY | historical_without_2025_evidence
PASS: TARGET_CURRENT | current_event_or_decision_evidence
PASS: TARGET_WITH_HISTORY | current_disaster_and_history
PASS: NEEDS_REVIEW | history_with_ambiguous_current_reference

EVENT SCOPE UNIT TEST: PASS


In [46]:
# 06C — EVENT SCOPE V2 (OFFLINE; V1 TIDAK DIUBAH)
# Letakkan setelah Cell 06B dan sebelum Cell 07.
# Membutuhkan: import re, html.unescape, match_any_phrase, DISASTER_CASES,
# get_audit_event_terms(), classify_event_scope() dari Cell 06B.

from html import unescape

EVENT_SCOPE_V2_VERSION = "event_scope_v2"


def _v2_normalize(value):
    """Normalisasi hanya untuk pemeriksaan; teks asli tetap disimpan."""
    return re.sub(r"\s+", " ", unescape(str(value or "")).casefold()).strip()


def _v2_re(text, pattern):
    return bool(re.search(pattern, text, flags=re.IGNORECASE))


def classify_event_scope_v2(text: str, case_id: str):
    """Klasifikasi heuristik konservatif; BUKAN gold-standard atau label sentimen.

    Return: (event_scope_v2, scope_reason_v2).
    """
    if case_id not in DISASTER_CASES:
        raise ValueError(f"Kasus tidak dikenali: {case_id}")

    sample = _v2_normalize(text)
    if not sample:
        return "NEEDS_REVIEW", "empty_text"

    # Periode lain tetap mengikuti V1; V2 kali ini difokuskan untuk
    # persoalan sejarah Aceh 2004 versus banjir Sumatra 2025.
    if case_id != "sumatra_2025":
        return classify_event_scope(sample, case_id)

    policy = match_any_phrase(sample, DISASTER_CASES[case_id]["policy_terms"])
    if not policy:
        return "NEEDS_REVIEW", "no_explicit_policy_term"

    locations = (
        "aceh", "sumatra", "sumatera", "sumut", "sumbar",
        "sumatera utara", "sumatra utara", "sumatera barat", "sumatra barat",
        "tapanuli", "tapteng", "taput", "tapsel", "medan", "agam",
        "aceh tamiang", "aceh utara", "langsa",
    )
    location = match_any_phrase(sample, list(locations))

    history = bool(
        _v2_re(sample, r"(?<!\w)(?:2004|1992|2006|2009|2020)(?!\w)")
        or _v2_re(sample, r"\btsunami\s+(?:aceh|sumatr[ae])\b")
        or _v2_re(sample, r"\bsumatr[ae][-\s]*andaman\b")
        or _v2_re(sample, r"\bcovid(?:[-\s]*19)?\b")
    )

    flood = _v2_re(sample, r"\b(?:banjir(?:\s+bandang)?|longsor|bencana\s+ekologis?)\b")
    explicit_year = _v2_re(sample, r"(?<!\w)2025(?!\w)")

    # Konteks tiga provinsi di Sumatra (termasuk ejaan nonbaku "propinsi").
    aceh = _v2_re(sample, r"\baceh\b")
    north = _v2_re(sample, r"\b(?:sumut|sumatr[ae]\s+utara|tapanuli|medan)\b")
    west = _v2_re(sample, r"\b(?:sumbar|sumatr[ae]\s+barat|padang|agam)\b")
    multi_region = (sum((aceh, north, west)) >= 3) or _v2_re(
        sample, r"\b(?:3|tiga)\s*(?:provinsi|propinsi)\b"
    )

    decision_action = _v2_re(
        sample,
        r"\b(?:belum|belom|blm|tidak|tdk|gak|ga|nggak|enggak|"
        r"bukan|tak\s+kunjung|mendesak|didesak|desak|minta|meminta|"
        r"dinaikkan|menaikkan|naikkan|naikin|tingkatkan|meningkatkan|"
        r"ditetapkan|menetapkan|tetapkan|penetapan|"
        r"kenapa|mengapa|perlukah|perlu|butuh|segera)\b"
    )
    now_cue = _v2_re(
        sample,
        r"\b(?:sekarang|skrg|skg|saat\s+ini|kini|hari\s+ini|"
        r"sampai\s+sekarang|hingga\s+kini|tahun\s+ini|"
        r"pemerintah\s+pusat\s+saat\s+ini|prabowo)\b"
    )
    contrasting = _v2_re(
        sample,
        r"\b(?:dulu|dlu|waktu\s+itu|zaman|jaman|"
        r"padahal|sedangkan|sementara|dibanding|beda|lah\s+ini|"
        r"kok|ko|sekarang|skrg)\b"
    )

    # Bukti kuat bahwa pembahasan menyasar bencana 2025.
    event_2025_evidence = location and (flood or explicit_year or multi_region)
    present_government = _v2_re(
        sample, r"\b(?:prabowo|pemerintah\s+pusat\s+saat\s+ini)\b"
    )
    policy_2025_evidence = location and decision_action and (multi_region or present_government)

    if history:
        if event_2025_evidence or policy_2025_evidence:
            return "TARGET_WITH_HISTORY", "2025_context_plus_history"
        if now_cue or contrasting or (location and decision_action and _v2_re(sample, r"\b(?:ini|tiga|3)\b")):
            return "NEEDS_REVIEW", "historical_comparison_ambiguous"
        return "HISTORICAL_ONLY", "historical_no_current_disaster_evidence"

    # Tanpa referensi sejarah, opini kebijakan di lokasi dan konteks
    # keputusan masa kini dapat tetap relevan meski tidak menyebut "banjir".
    if event_2025_evidence:
        return "TARGET_CURRENT", "2025_event_evidence"
    if location and decision_action:
        return "TARGET_CURRENT", "location_and_policy_decision"
    return "NEEDS_REVIEW", "insufficient_event_context"


# Unit test dari tweet nyata yang sebelumnya diperiksa.
# Ekspektasi ini adalah adjudikasi sementara untuk regresi, bukan skor akurasi.
V2_REGRESSION_SAMPLES = [
    (
        "1996005502111846540",
        "Ya Allah udh 7 hari kelaparan. Makanya rakyat minta ditetapkan bencana nasional "
        "Tsunami aceh aja dlu cm satu propinsi bisa jd bencana nasional, skrg 3 propinsi",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1996594403432112336",
        "Kalo zaman SBY, hari ke 3 setelah bencana tsunami Aceh langsung tetapkan "
        "status bencana nasional. Pemerintah pusat saat ini hanya sekumpulan "
        "pengangguran yg mencari uang melalui kekuasaan @prabowo",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1996207496550318418",
        "Sekarang dpr sudah mendesak status bencana nasional. Waktu tsunami aceh "
        "bisa ko, lah ini? 3 provinsi masa dibiarin",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1994790454051713494",
        "Polemik Status Bencana Nasional: Perlukah Sumatera dan Aceh Dinaikkan?",
        "TARGET_CURRENT",
    ),
    (
        "1996227070318285049",
        "Saat ini berharap dinaikkan status bencana kami ini ke bencana nasional "
        "apalagi Aceh, Tapanuli, dan Sumbar",
        "TARGET_CURRENT",
    ),
    (
        "1996027810281128040",
        "Keuangan Pemda ACEH-SUMUT dan SUMBAR tidak akan mampu mengatasinya. "
        "Presiden umumkan bencana ini sebagai BENCANA NASIONAL",
        "TARGET_CURRENT",
    ),
    (
        "1996419143890018743",
        "Menurut pemerintahan Prabowo bencana Aceh Sumut Sumbar BUKAN Bencana Nasional",
        "TARGET_CURRENT",
    ),
    (
        "1996074461297344839",
        "Gempa tsunami Aceh 2004, banjir bandang longsor Aceh Sumut Sumbar 2025 "
        "mana yang diakui bencana nasional?",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1996352009776304267",
        "Gempa tsunami Sumatra-Andaman 2004 dinyatakan sbg bencana nasional "
        "setelah 12 jam, jalur komunikasi Aceh terputus",
        "HISTORICAL_ONLY",
    ),
    (
        "1996265317547520148",
        "Tsunami Aceh sudah ditetapkan bencana nasional, momentum penyelesaian "
        "konflik dan perjanjian Helsinki",
        "HISTORICAL_ONLY",
    ),
    (
        "manual_ambigu",
        "Aceh 2004 dulu bencana nasional, sekarang mengapa status itu masih diperdebatkan?",
        "NEEDS_REVIEW",
    ),
    (
        "manual_non_case",
        "Status bencana nasional mungkin mengundang bantuan asing",
        "NEEDS_REVIEW",
    ),
]

for tweet_id, sample_text, expected in V2_REGRESSION_SAMPLES:
    result, reason = classify_event_scope_v2(sample_text, "sumatra_2025")
    assert result == expected, (
        f"Tweet {tweet_id}: expected {expected}; got {result} ({reason})"
    )

print(f"EVENT SCOPE V2 TEST: PASS ({len(V2_REGRESSION_SAMPLES)} kasus; tanpa Apify)")


EVENT SCOPE V2 TEST: PASS (12 kasus; tanpa Apify)


In [47]:
# 06D — AUDIT FOKUS PENETAPAN STATUS NASIONAL (V2.1, OFFLINE)
# Sisipkan setelah Cell 06C. Biarkan fungsi dan laporan V1/V2 apa adanya.
# Ini menilai fokus kebijakan, BUKAN mengganti event_scope V2 atau label manual.

import re
from html import unescape

POLICY_FOCUS_VERSION = "policy_focus_v2_1"


def _pf21_norm(value):
    return re.sub(r"\s+", " ", unescape(str(value or "")).casefold()).strip()


def _pf21_has(text, regex):
    return bool(re.search(regex, text, flags=re.IGNORECASE))


def classify_policy_focus_v21(text, case_id, v2_scope=""):
    """Return (policy_focus_v21, focus_reason_v21).

    INCLUDE_CANDIDATE = berbicara tentang penetapan/penolakan status nasional
                        dan memiliki bukti peristiwa target.
    EXCLUDE_CANDIDATE = tak membahas keputusan status / bukan peristiwa target.
    NEEDS_CONTEXT     = perlu konteks thread / anotasi manual, jangan dibuang.
    """
    t = _pf21_norm(text)
    if not t:
        return "NEEDS_CONTEXT", "empty_text"
    if case_id not in ("lombok_2018", "palu_2018", "sumatra_2025"):
        raise ValueError(f"Unknown case: {case_id}")

    national_status = _pf21_has(
        t,
        r"\b(?:bencana\s+nasional|status\s+(?:bencana\s+)?nasional|"
        r"status\s+bencana|status\s+darurat)\b",
    )
    decision_action = _pf21_has(
        t,
        r"\b(?:status|penetap\w*|menetap\w*|ditetap\w*|tetap\w*|"
        r"netap\w*|netep\w*|menaik\w*|dinaik\w*|naikkan|naikin|"
        r"ditingkat\w*|meningkat\w*|menjadi|dijadi\w*|jadikan|"
        r"dinyata\w*|menyata\w*|deklarasi|dideklarasi\w*|diumum\w*|"
        r"menolak|ditolak|bukan|belum|belom|tidak\s+masuk\s+kriteria|"
        r"ga\s+jadi|gak\s+jadi|nggak\s+jadi)\b",
    )
    # Konteks penolakan eksplisit dengan gaya informal: "bencana nasional, ya ga lah".
    denial_after_status = _pf21_has(
        t,
        r"\bbencana\s+nasional\W{0,8}(?:ya\s+)?(?:ga|gak|nggak|enggak|tidak|bukan)\b",
    )
    if not national_status:
        return "EXCLUDE_CANDIDATE", "no_national_status_discussion"
    # Berita bencana semata bukan pembahasan penetapan status; pengecualian
    # eksplisit berikut menghindari homonim/metafora bencana nasional.
    if _pf21_has(t, r"\bkorupsi\b.{0,60}\bbencana\s+nasional\b") and not decision_action:
        return "EXCLUDE_CANDIDATE", "metaphor_not_national_disaster_decision"
    if _pf21_has(t, r"\bmemimpin\s+penangg?ulangan\s+bencana\s+nasional\b") and not decision_action:
        return "EXCLUDE_CANDIDATE", "leadership_of_response_not_status_decision"
    if not (decision_action or denial_after_status):
        # 'bencana nasional' tanpa verba penetapan tidak cukup untuk
        # otomatis ditolak: bisa berupa diskusi status atau judul artikel.
        return "NEEDS_CONTEXT", "national_disaster_phrase_without_clear_decision"

    if case_id != "sumatra_2025":
        terms = (
            ["Lombok", "NTB", "Nusa Tenggara Barat"] if case_id == "lombok_2018"
            else ["Palu", "Donggala", "Sulteng", "Sulawesi Tengah"]
        )
        at_target = any(_pf21_has(t, r"(?<!\w)" + re.escape(x.casefold()) + r"(?!\w)") for x in terms)
        if at_target:
            return "INCLUDE_CANDIDATE", "explicit_target_and_status_decision"
        return "NEEDS_CONTEXT", "national_status_without_target_event"

    # Aceh 2004 tidak boleh otomatis diartikan sebagai banjir/longsor 2025.
    historical = _pf21_has(
        t,
        r"(?<!\w)(?:2004|1992|2006|2009|2020)(?!\w)|"
        r"\b(?:tsunami\s+aceh|tsunami\s+sumatr[ae]|sumatr[ae][\s-]*andaman|covid)\b",
    )
    location = _pf21_has(
        t,
        r"\b(?:aceh|sumatr[ae]|sumut|sumbar|tapanuli|tapteng|taput|"
        r"tapsel|medan|padang|agam|langsa)\b",
    )
    present_disaster = _pf21_has(
        t, r"\b(?:banjir|longsor|banjir\s+bandang|bencana\s+ekologis?)\b"
    )
    year_target = _pf21_has(t, r"(?<!\w)2025(?!\w)")
    aceh = _pf21_has(t, r"\baceh\b")
    north = _pf21_has(t, r"\b(?:sumut|sumatr[ae]\s+utara|tapanuli|medan)\b")
    west = _pf21_has(t, r"\b(?:sumbar|sumatr[ae]\s+barat|padang|agam)\b")
    multiregion = (int(aceh) + int(north) + int(west) >= 2) or _pf21_has(
        t, r"\b(?:3|tiga)\s*(?:provinsi|propinsi)\b"
    )
    present_cue = _pf21_has(
        t, r"\b(?:sekarang|skrg|skg|saat ini|hingga kini|hari ini|tahun ini|lah ini)\b"
    )
    # Dua provinsi Aceh+Sumut pada uraian tsunami 2004 tidak otomatis
    # membuktikan ada pembahasan banjir 2025.
    target_evidence = location and (
        present_disaster or year_target
        or (multiregion and (not historical or present_cue))
    )
    if target_evidence:
        return "INCLUDE_CANDIDATE", (
            "target_2025_with_historical_comparison" if historical
            else "target_2025_status_decision"
        )
    if historical:
        # Daftar statis kejadian lama tanpa target 2025 / ajakan membandingkan.
        if _pf21_has(t, r"\b(?:1992|2004)\b") and _pf21_has(t, r"\b2020\b") and not _pf21_has(
            t, r"\b(?:sekarang|skrg|saat ini|kini|lah ini|masa|kok|ko|padahal|banding)\b"
        ):
            return "EXCLUDE_CANDIDATE", "static_history_list_no_target_2025"
        # Beberapa tweet membandingkan pemerintah masa kini vs 2004, tetapi
        # tidak menyebut banjir 2025; jangan paksa INCLUDE maupun EXCLUDE.
        return "NEEDS_CONTEXT", "historical_without_explicit_2025_event"
    if location:
        # Dalam arsip 2025 hal ini mungkin terkait bencana target, tetapi
        # membutuhkan konteks lebih kuat sebelum diperlakukan sebagai fakta.
        return "NEEDS_CONTEXT", "place_and_status_without_2025_event_anchor"
    return "NEEDS_CONTEXT", "status_without_identifiable_target_event"


# UJI REGRESI — teks asli 6 kasus review vs V2 (bukan pelatihan model).
# Dua 'manual INCLUDE' historis sengaja diarahkan ke NEEDS_CONTEXT,
# bukan dipaksa menjadi peristiwa Sumatra 2025.
POLICY_FOCUS_V21_TESTS = [
    ("lombok_2018", "Korupsi itu baru bencana nasional kl yg di Lombok itu bencana alam", "EXCLUDE_CANDIDATE"),
    ("palu_2018", "Presiden harus memimpin penanggulangan Bencana Nasional di Palu", "EXCLUDE_CANDIDATE"),
    ("lombok_2018", "Gempa kembali mengguncang Lombok dan korban meninggal", "EXCLUDE_CANDIDATE"),
    ("sumatra_2025", "Bencana nasional, ya ga lah. Tsunami Aceh dan gempa Yogya dulu", "NEEDS_CONTEXT"),
    ("sumatra_2025", "Pemerintah netapin bencana nasional baru tsunami Aceh dan covid; bencana Palu tidak", "NEEDS_CONTEXT"),
    ("sumatra_2025", "Waktu tsunami Aceh SBY tetapkan status bencana nasional; pemerintah pusat saat ini berbeda", "NEEDS_CONTEXT"),
    ("sumatra_2025", "Banjir Aceh Sumut Sumbar 2025 belum ditetapkan bencana nasional", "INCLUDE_CANDIDATE"),
    ("palu_2018", "PKS desak status bencana nasional bagi gempa Palu", "INCLUDE_CANDIDATE"),
]
for _case, _text, _expected in POLICY_FOCUS_V21_TESTS:
    _got, _reason = classify_policy_focus_v21(_text, _case)
    assert _got == _expected, (_case, _expected, _got, _reason)
print(f"POLICY FOCUS V2.1: PASS ({len(POLICY_FOCUS_V21_TESTS)} skenario; offline)")


POLICY FOCUS V2.1: PASS (8 skenario; offline)


## Langkah 4 — Penyimpanan raw dan checkpoint
Satu folder per kasus. JSONL mempertahankan objek tweet original, audit CSV adalah hasil turunan.
Checkpoint memuat hash task (mempertimbangkan query dan waktu), run ID, dataset ID dan indikator kualitas.
Task yang sudah tersimpan tidak dipanggil ulang saat notebook dijalankan ulang.

In [48]:
# 07 — HELPER CHECKPOINT DAN PENYIMPANAN

def json_write_atomic(path: Path, items: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False, default=str) + '\n')
    tmp.replace(path)


def csv_write_atomic(path: Path, df: pd.DataFrame):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, index=False, encoding='utf-8-sig')
    tmp.replace(path)


def append_checkpoint(record: dict):
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with CHECKPOINT_FILE.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        f.flush()
        os.fsync(f.fileno())


def load_checkpoint() -> dict[str, dict]:
    newest = {}
    if not CHECKPOINT_FILE.exists():
        return newest
    with CHECKPOINT_FILE.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError('Checkpoint rusak. Periksa file sebelum lanjut.') from exc
            newest[entry['task_id']] = entry
    return newest


def make_run_record(task: dict, run_dict: dict, metrics: dict,
                    flags: list[str], raw_path: Path, audit_path: Path) -> dict:
    # SAVED = berhasil disimpan; data_quality_status = ada atau tidak kandidat.
    data_quality_status = (
        'NO_REAL_DATA' if metrics['unique_real_ids'] == 0 else
        'NO_POLICY_CANDIDATES' if metrics['unique_policy_candidate_ids'] == 0 else
        'HAS_POLICY_CANDIDATES'
    )
    return {
        'task_id': task['task_id'], 'config_version': CONFIG_VERSION,
        'status': 'SAVED', 'data_quality_status': data_quality_status,
        'case': task['case'], 'query_id': task['query_id'],
        'depth': task['depth'], 'actor_id': ACTOR_ID,
        'run_id': run_dict.get('id'),
        'dataset_id': run_dict.get('defaultDatasetId') or run_dict.get('default_dataset_id'),
        'actor_status': run_dict.get('status'),
        'query': task['query'], 'base_query': task['base_query'],
        'start_wib': iso_wib(task['start_wib']),
        'end_wib': iso_wib(task['end_wib']),
        'start_utc': task['start_wib'].astimezone(UTC).isoformat(),
        'end_utc': task['end_wib'].astimezone(UTC).isoformat(),
        'quality_flags': flags, **metrics,
        'raw_path': str(raw_path.relative_to(PROJECT_ROOT)),
        'audit_path': str(audit_path.relative_to(PROJECT_ROOT)),
        'saved_at_utc': datetime.now(UTC).isoformat(),
        'usage_total_usd': run_dict.get('usageTotalUsd', run_dict.get('usage_total_usd')),
        'max_total_charge_usd_requested': str(task.get('per_run_cap_usd', '')),
        'server_max_items_requested': MAX_ITEMS,
    }


def run_to_dict(run: Any) -> dict:
    if isinstance(run, dict):
        return run
    if hasattr(run, 'model_dump'):
        return run.model_dump(by_alias=True)
    if hasattr(run, 'to_dict'):
        return run.to_dict()
    if hasattr(run, 'default_dataset_id'):
        return {
            'default_dataset_id': run.default_dataset_id,
            'id': getattr(run, 'id', None),
            'status': getattr(run, 'status', None),
        }
    raise TypeError(f'Tipe Apify Run tidak dikenal: {type(run)}')


def get_apify_client() -> ApifyClient:
    token = os.getenv('APIFY_API_TOKEN', '').strip()
    if not token:
        token = getpass('Apify API token (tidak tampil): ').strip()
    if not token:
        raise RuntimeError('API token kosong. Tidak dapat menjalankan Actor.')
    # Jangan cetak atau simpan token dalam notebook, checkpoint, atau GitHub.
    return ApifyClient(token)


In [49]:
# 07A — PEMERIKSAAN BIAYA HISTORIS (READ-ONLY, OPSIONAL; DEFAULT OFF)
# Mengaktifkan ini hanya membaca metadata run lama. Tidak memulai scraping baru.
CHECK_HISTORICAL_COSTS = False

if CHECK_HISTORICAL_COSTS:
    _history_client = get_apify_client()
    _cost_rows = []
    for rec in load_checkpoint().values():
        if rec.get('status') != 'SAVED' or not rec.get('run_id'):
            continue
        info = run_to_dict(_history_client.run(rec['run_id']).get() or {})
        price = info.get('usageTotalUsd', info.get('usage_total_usd'))
        _cost_rows.append({
            'case': rec.get('case'), 'run_id': rec.get('run_id'),
            'rows_raw': rec.get('rows_raw'),
            'unique_ids_in_run': rec.get('unique_real_ids'),
            'usage_total_usd': price,
        })
    df_cost = pd.DataFrame(_cost_rows)
    display(df_cost)
    if not df_cost.empty:
        _charges = pd.to_numeric(df_cost['usage_total_usd'], errors='coerce')
        print('Biaya run dengan data tersedia: $', round(_charges.sum(), 5))
        print('Run tanpa data biaya:', int(_charges.isna().sum()))
        print('Jangan menganggap NaN sebagai biaya $0.')
else:
    print('Pemeriksaan biaya historis dilewati (tidak ada API call).')


Pemeriksaan biaya historis dilewati (tidak ada API call).


In [50]:
# 07B — REKONSILIASI BIAYA SMOKE TEST (OFFLINE, TANPA TOKEN API)
# Periksa: Apify Billing > Usage menampilkan $1.21 (sebelum), $1.22 (sesudah).
# Selisih dashboard adalah pemakaian PLATFORM, bukan bukti tagihan per Actor run.
USAGE_BEFORE_SMOKE_USD = Decimal('1.21')
USAGE_AFTER_SMOKE_USD = Decimal('1.22')
SMOKE_RUN_IDS = (
    'Q02DRjtfM0hRo9D5h',  # Lombok
    'NJ4LiyFPaZlHXA1wd',  # Palu
    'htS8vz5ELBdk9BGTH',  # Sumatra
)

_state_cost = load_checkpoint()
_smoke = [r for r in _state_cost.values()
          if r.get('run_id') in SMOKE_RUN_IDS and r.get('status') == 'SAVED']
assert len(_smoke) == 3, 'Run smoke test belum lengkap pada checkpoint lokal.'

df_smoke_cost = pd.DataFrame([{
    'case': r['case'],
    'run_id': r['run_id'],
    'raw': r.get('rows_raw', 0),
    'real': r.get('rows_real_tweet', 0),
    'mock': r.get('rows_mock', 0),
    'reported_run_usage_usd': r.get('usage_total_usd'),
} for r in _smoke])
display(df_smoke_cost)

_delta = USAGE_AFTER_SMOKE_USD - USAGE_BEFORE_SMOKE_USD
_run_reported = sum(Decimal(str(r.get('usage_total_usd') or 0)) for r in _smoke)
print('Selisih penggunaan platform dari dashboard: $', _delta)
print('Jumlah usage_total_usd pada checkpoint   : $', _run_reported)
if _delta != _run_reported:
    print('PERINGATAN: angka platform dan metadata run tidak sama. '
          'Jangan menafsirkan 0 pada checkpoint sebagai gratis. '
          'Cocokkan rincian Apify Billing/Usage sebelum melanjutkan.')
print('Tidak ada panggilan Actor baru pada cell ini.')


,case,run_id,raw,real,mock,reported_run_usage_usd
0,lombok_2018,Q02DRjtfM0hRo9D5h,15,0,15,0.0
1,palu_2018,NJ4LiyFPaZlHXA1wd,15,0,15,0.0
2,sumatra_2025,htS8vz5ELBdk9BGTH,15,0,15,0.0


Selisih penggunaan platform dari dashboard: $ 0.01
Jumlah usage_total_usd pada checkpoint   : $ 0
PERINGATAN: angka platform dan metadata run tidak sama. Jangan menafsirkan 0 pada checkpoint sebagai gratis. Cocokkan rincian Apify Billing/Usage sebelum melanjutkan.
Tidak ada panggilan Actor baru pada cell ini.


In [51]:
# 07C — GENERIC NEXT-RUN PREVIEW (OFFLINE)
# Berlaku untuk setiap batch, tanpa hard-coded tanggal.

def get_next_tasks(limit=None):
    assert RUN_MODE == "full"
    assert ENABLE_ADAPTIVE is False

    limit = MAX_NEW_RUNS if limit is None else limit
    assert 1 <= limit <= 3

    state = load_checkpoint()
    tasks = build_tasks("full")

    known_statuses = {
        "SAVED", "RESERVED", "NEEDS_REVIEW",
        "FAILED", "FAILED_NEEDS_REVIEW"
    }

    # Fail closed jika checkpoint punya status tak dikenal.
    for task in tasks:
        rec = state.get(task["task_id"])

        if rec and rec.get("status") not in known_statuses:
            raise RuntimeError(
                f"Status tidak dikenal: {task['task_id']}"
            )

    # Hanya task yang benar-benar belum tercatat.
    pending = [
        task for task in tasks
        if task["task_id"] not in state
    ]

    return pending[:limit]


def preview_next_runs(limit=None):
    tasks = get_next_tasks(limit)
    state = load_checkpoint()

    print("=" * 65)
    print("GENERIC NEXT-RUN PREVIEW")
    print("Mode:", RUN_MODE)
    print("Scraping enabled:", EXECUTE_SCRAPING)
    print("Max run:", MAX_NEW_RUNS)
    print("Saved tasks:", sum(
        r.get("status") == "SAVED"
        for r in state.values()
    ))

    print("\nNEXT TASKS")

    for i, task in enumerate(tasks, 1):
        print(
            f"{i}. {task['case']} "
            f"{task['query_id']}"
        )
        print(
            "  ",
            iso_wib(task["start_wib"]),
            "→",
            iso_wib(task["end_wib"])
        )
        print("   Task ID:", task["task_id"])

    if not tasks:
        print("Tidak ada task pending.")

    print("\nPREVIEW COMPLETE — NO API CALL")
    return tasks


# Simpan identitas task yang telah diperiksa.
REVIEWED_NEXT_TASK_IDS = tuple(
    task["task_id"]
    for task in preview_next_runs()
)


GENERIC NEXT-RUN PREVIEW
Mode: full
Scraping enabled: False
Max run: 1
Saved tasks: 59

NEXT TASKS
1. lombok_2018 E01_P01
   2018-07-29 18:00:00+0700 → 2018-07-30 00:00:00+0700
   Task ID: lombok_2018_87720489a7da293c7ba3

PREVIEW COMPLETE — NO API CALL


In [76]:
# 07D.0 — BATCH 7 HISTORICAL YIELD REVIEW
# OFFLINE ONLY — UPDATED THROUGH PART 11
# ============================================================

import pandas as pd
from IPython.display import display

assert EXECUTE_SCRAPING is False, "Scraping harus OFF."
assert PAID_RUN_CONFIRMATION == "", "Paid gate harus LOCKED."

state = load_checkpoint()

if any(rec.get("status") != "SAVED" for rec in state.values()):
    raise RuntimeError(
        "Ada task belum SAVED. Periksa checkpoint terlebih dahulu."
    )

# Histori Batch 7 yang sudah dijalankan
BATCH7_HISTORY = {
    "lombok_2018_cd32f2d935983c722c52": "Part 1",
    "palu_2018_d89e4b7de6aab92d1fc4": "Part 2",
    "lombok_2018_1cedb1f81b64ae640f98": "Part 3",
    "palu_2018_cef075f5f909c22c3fee": "Part 4",
    "palu_2018_49123b6447743a858b7b": "Part 5",
    "palu_2018_4aaecc325267efb249b6": "Part 6",
    "lombok_2018_5b80485fc9f119e18517": "Part 8",
    "palu_2018_5fc578fbe346020d0eeb": "Part 9",
    "lombok_2018_259456e255c90766b223": "Part 10",
    "lombok_2018_2d2f428d9fd140d30a4e": "Part 11",
}

rows = []

for rec in state.values():
    raw = int(rec.get("rows_raw") or 0)
    real = int(rec.get("rows_real_tweet") or 0)
    mock = int(rec.get("rows_mock") or 0)
    unique = int(rec.get("unique_real_ids") or 0)

    if raw != real + mock:
        raise RuntimeError(
            f"Metrik tidak cocok: {rec['task_id']}"
        )

    rows.append({
        "case": rec["case"],
        "task_id": rec["task_id"],
        "query_id": rec.get("query_id", ""),
        "start_wib": rec.get("start_wib", ""),
        "end_wib": rec.get("end_wib", ""),
        "raw": raw,
        "real_rows": real,
        "mock_rows": mock,
        "unique_ids_in_run": unique,
        "mock_pct": round(100 * mock / raw, 1) if raw else 0,
        "data_quality_status": rec.get("data_quality_status", ""),
    })

df_yield = pd.DataFrame(rows)

missing_tasks = set(BATCH7_HISTORY) - set(df_yield["task_id"])

if missing_tasks:
    raise RuntimeError(
        f"Histori Batch 7 belum lengkap: {sorted(missing_tasks)}"
    )

recent = df_yield.loc[
    df_yield["task_id"].isin(BATCH7_HISTORY)
].copy()

recent["batch"] = recent["task_id"].map(BATCH7_HISTORY)

recent["part_order"] = (
    recent["batch"].str.extract(r"(\d+)")[0].astype(int)
)

recent = recent.sort_values("part_order")

EXPECTED_BATCH7_RUNS = 10

if len(recent) != EXPECTED_BATCH7_RUNS:
    raise RuntimeError(
        f"Expected {EXPECTED_BATCH7_RUNS} runs, "
        f"found {len(recent)}."
    )

print("=" * 65)
print("BATCH 7 HISTORICAL YIELD REVIEW")
print("=" * 65)

display(
    recent[
        [
            "batch", "case", "raw",
            "real_rows", "mock_rows",
            "unique_ids_in_run", "mock_pct"
        ]
    ]
)

print("\nRUN PALING PRODUKTIF SECARA HISTORIS")

display(
    df_yield.sort_values(
        ["unique_ids_in_run", "real_rows"],
        ascending=False
    ).head(12)[
        [
            "case", "start_wib", "end_wib",
            "unique_ids_in_run",
            "real_rows", "mock_pct"
        ]
    ]
)

print("\nRINGKASAN GLOBAL")
print("Task SAVED:", len(state))
print("RAW total:", int(df_yield["raw"].sum()))
print("Real rows:", int(df_yield["real_rows"].sum()))
print("Mock rows:", int(df_yield["mock_rows"].sum()))
print("Histori Batch 7:", len(recent))
print("API calls baru: 0")
print("=" * 65)

BATCH 7 HISTORICAL YIELD REVIEW


,batch,case,raw,real_rows,mock_rows,unique_ids_in_run,mock_pct
49,Part 1,lombok_2018,22,14,8,7,36.4
50,Part 2,palu_2018,38,38,0,19,0.0
51,Part 3,lombok_2018,16,2,14,1,87.5
52,Part 4,palu_2018,16,2,14,1,87.5
53,Part 5,palu_2018,24,18,6,9,25.0
54,Part 6,palu_2018,15,4,11,4,73.3
55,Part 8,lombok_2018,20,10,10,5,50.0
56,Part 9,palu_2018,17,4,13,2,76.5
57,Part 10,lombok_2018,21,12,9,6,42.9
58,Part 11,lombok_2018,19,8,11,4,57.9



RUN PALING PRODUKTIF SECARA HISTORIS


,case,start_wib,end_wib,unique_ids_in_run,real_rows,mock_pct
14,sumatra_2025,2025-12-03 06:00:00+0700,2025-12-03 12:00:00+0700,27,35,0.0
17,sumatra_2025,2025-12-03 12:00:00+0700,2025-12-03 18:00:00+0700,21,40,0.0
2,sumatra_2025,2025-11-29 00:00:00+0700,2025-11-30 00:00:00+0700,20,40,0.0
6,lombok_2018,2018-08-13 00:00:00+0700,2018-08-14 00:00:00+0700,20,40,0.0
7,palu_2018,2018-10-02 00:00:00+0700,2018-10-03 00:00:00+0700,20,40,0.0
8,sumatra_2025,2025-12-03 00:00:00+0700,2025-12-04 00:00:00+0700,20,40,0.0
21,sumatra_2025,2025-12-03 09:00:00+0700,2025-12-03 15:00:00+0700,20,40,0.0
43,sumatra_2025,2025-12-02 18:00:00+0700,2025-12-03 00:00:00+0700,20,40,0.0
47,palu_2018,2018-10-01 12:00:00+0700,2018-10-01 18:00:00+0700,20,40,0.0
50,palu_2018,2018-10-01 18:00:00+0700,2018-10-02 00:00:00+0700,19,38,0.0



RINGKASAN GLOBAL
Task SAVED: 60
RAW total: 1401
Real rows: 925
Mock rows: 476
Histori Batch 7: 10
API calls baru: 0


In [53]:
# 07D — BATCH 7 DYNAMIC ONE-RUN PREVIEW
# OFFLINE ONLY | LOMBOK + PALU PRIORITY
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime
from decimal import Decimal

TARGET_RAW = 1500
TARGET_UNIQUE_IDS = 400

BATCH7_DIR = REPORT_DIR / "batch7"
BATCH7_BASELINE_FILE = (
    BATCH7_DIR / "baseline_manifest.json"
)

# Jumlah baris checkpoint pada commit pre batch7.
BASELINE_CHECKPOINT_LINES = 74

def batch7_scan_saved():
    """
    Audit read-only terhadap checkpoint dan seluruh RAW.
    Dipakai bersama oleh Cell 07D, 09D, 11B.1, 11H.
    """

    if not CHECKPOINT_FILE.is_file():
        raise FileNotFoundError(CHECKPOINT_FILE)

    if not BATCH7_BASELINE_FILE.is_file():
        raise FileNotFoundError(
            "Snapshot baseline tidak ditemukan."
        )

    with BATCH7_BASELINE_FILE.open(
        encoding="utf-8"
    ) as f:
        baseline = json.load(f)

    expected_baseline = {
        "tasks": 49,
        "raw": 1171,
        "tweet_rows": 799,
        "mock_rows": 372,
        "unique_ids": 314,
    }

    if baseline.get("counts") != expected_baseline:
        raise RuntimeError(
            "Baseline manifest bukan hasil pre batch7."
        )

    checkpoint_bytes = CHECKPOINT_FILE.read_bytes()
    checkpoint_lines = checkpoint_bytes.splitlines(
        keepends=True
    )

    if len(checkpoint_lines) < BASELINE_CHECKPOINT_LINES:
        raise RuntimeError(
            "Riwayat checkpoint lebih pendek dari baseline."
        )

    baseline_prefix = b"".join(
        checkpoint_lines[:BASELINE_CHECKPOINT_LINES]
    )

    baseline_prefix_sha = hashlib.sha256(
        baseline_prefix
    ).hexdigest()

    if baseline_prefix_sha != baseline[
        "checkpoint_sha256"
    ]:
        raise RuntimeError(
            "Riwayat checkpoint lama berubah. STOP."
        )

    state = load_checkpoint()

    if len(state) < 49:
        raise RuntimeError(
            "Jumlah task turun dari baseline."
        )

    if any(
        rec.get("status") != "SAVED"
        for rec in state.values()
    ):
        raise RuntimeError(
            "Ada task RESERVED / NEEDS_REVIEW / FAILED. "
            "Periksa sebelum run baru."
        )

    baseline_raw = {
        r["task_id"]: r
        for r in baseline["file_manifest"]
    }

    if len(baseline_raw) != 49:
        raise RuntimeError(
            "Manifest 49 file RAW tidak lengkap."
        )

    by_case = {
        case: {
            "raw": 0,
            "tweet_rows": 0,
            "mock_rows": 0,
            "ids": set(),
            "tasks": 0,
        }
        for case in FULL_CASES
    }

    ids_by_task = {}
    global_ids = set()
    id_owner = {}

    total_raw = 0
    total_tweet = 0
    total_mock = 0

    raw_manifest_hash = hashlib.sha256()

    for task_id in sorted(state):

        rec = state[task_id]
        case = rec.get("case")

        if case not in by_case:
            raise RuntimeError(
                f"Kasus tidak dikenal: {case}"
            )

        rel = str(
            rec.get("raw_path") or ""
        ).replace("\\", "/")

        audit_rel = str(
            rec.get("audit_path") or ""
        ).replace("\\", "/")

        raw_path = PROJECT_ROOT / Path(rel)
        audit_path = PROJECT_ROOT / Path(audit_rel)

        if not raw_path.is_file():
            raise FileNotFoundError(raw_path)

        if not audit_path.is_file():
            raise FileNotFoundError(audit_path)

        raw_bytes = raw_path.read_bytes()
        raw_sha = hashlib.sha256(
            raw_bytes
        ).hexdigest()

        # Lindungi 49 RAW baseline dari perubahan.
        if task_id in baseline_raw:
            old = baseline_raw[task_id]

            if (
                raw_sha != old["sha256"]
                or rel != old["raw_path"].replace("\\", "/")
            ):
                raise RuntimeError(
                    "RAW baseline berubah: " + task_id
                )

        raw_manifest_hash.update(
            task_id.encode("utf-8")
        )
        raw_manifest_hash.update(
            raw_sha.encode("ascii")
        )

        n_raw = 0
        n_real = 0
        n_mock = 0
        task_ids = set()

        for line in raw_bytes.decode(
            "utf-8"
        ).splitlines():

            if not line.strip():
                continue

            item = json.loads(line)

            n_raw += 1
            item_type = item.get("type")

            if item_type == "mock_tweet":
                n_mock += 1
                continue

            if item_type != "tweet":
                raise RuntimeError(
                    "Tipe record tidak dikenal: "
                    + task_id
                )

            n_real += 1

            tid = str(
                item.get("id", "")
            ).strip()

            if tid in ("", "-1", "None"):
                raise RuntimeError(
                    "ID tweet tidak valid: " + task_id
                )

            previous_case = id_owner.get(tid)

            if previous_case and previous_case != case:
                raise RuntimeError(
                    "ID ditemukan lintas kasus: "
                    + tid
                    + ". Perlu audit alokasi kasus."
                )

            id_owner[tid] = case

            task_ids.add(tid)
            global_ids.add(tid)
            by_case[case]["ids"].add(tid)

        if (
            n_raw != rec.get("rows_raw")
            or n_real != rec.get("rows_real_tweet")
            or n_mock != rec.get("rows_mock")
        ):
            raise RuntimeError(
                "Metrik checkpoint berbeda dari RAW: "
                + task_id
            )

        ids_by_task[task_id] = task_ids

        total_raw += n_raw
        total_tweet += n_real
        total_mock += n_mock

        by_case[case]["raw"] += n_raw
        by_case[case]["tweet_rows"] += n_real
        by_case[case]["mock_rows"] += n_mock
        by_case[case]["tasks"] += 1

    if not set(baseline_raw).issubset(state):
        raise RuntimeError(
            "Ada task baseline yang hilang."
        )

    if (
        total_raw < 1171
        or total_tweet < 799
        or total_mock < 372
        or len(global_ids) < 314
    ):
        raise RuntimeError(
            "Jumlah data lebih kecil dari baseline."
        )

    return {
        "state": state,
        "raw_total": total_raw,
        "tweet_rows": total_tweet,
        "mock_rows": total_mock,
        "unique_total": len(global_ids),
        "global_ids": global_ids,
        "ids_by_task": ids_by_task,
        "by_case": by_case,
        "checkpoint_sha": hashlib.sha256(
            checkpoint_bytes
        ).hexdigest(),
        "raw_manifest_sha": (
            raw_manifest_hash.hexdigest()
        ),
    }


# ------------------------------------------------------------
# A. PASTIKAN MODE OFFLINE
# ------------------------------------------------------------

if EXECUTE_SCRAPING is not False:
    raise RuntimeError(
        "Cell 07D hanya boleh dijalankan OFFLINE."
    )

if PAID_RUN_CONFIRMATION != "":
    raise RuntimeError(
        "Konfirmasi berbayar harus kosong."
    )

if (
    RUN_MODE != "full"
    or MAX_NEW_RUNS != 1
    or MAX_ITEMS != 20
    or ENABLE_ADAPTIVE
    or RETRY_UNCERTAIN_TASKS
):
    raise RuntimeError(
        "Konfigurasi Batch 7 tidak sesuai."
    )

if (
    Decimal(str(MAX_SESSION_SPEND_USD))
    != Decimal("0.03")
    or Decimal(str(MAX_CHARGE_PER_RUN_USD))
    != Decimal("0.03")
):
    raise RuntimeError(
        "Budget harus US$0.03 per sesi/run."
    )


# ------------------------------------------------------------
# B. BACA KONDISI DATA TERBARU
# ------------------------------------------------------------

snapshot = batch7_scan_saved()

state = snapshot["state"]

CURRENT_RAW_TOTAL = snapshot["raw_total"]
CURRENT_UNIQUE_TOTAL = snapshot["unique_total"]

print("RAW sekarang:", CURRENT_RAW_TOTAL)
print("ID unik sekarang:", CURRENT_UNIQUE_TOTAL)

print(
    "Kurang RAW:",
    max(0, TARGET_RAW - CURRENT_RAW_TOTAL)
)

print(
    "Kurang ID unik:",
    max(0, TARGET_UNIQUE_IDS - CURRENT_UNIQUE_TOTAL)
)

if (
    CURRENT_RAW_TOTAL >= TARGET_RAW
    and CURRENT_UNIQUE_TOTAL >= TARGET_UNIQUE_IDS
):
    raise RuntimeError(
        "DUA TARGET SUDAH TERCAPAI. "
        "STOP scraping dan lanjutkan analisis."
    )


# ------------------------------------------------------------
# C. PRIORITAS WINDOW BARU
# ------------------------------------------------------------
# Urutan dipilih untuk keseimbangan Lombok/Palu.
# Setiap task dicek terhadap seluruh histori SAVED.


# ------------------------------------------------------------
# PRIORITY WINDOWS — BATCH 7 PART 5
# Kandidat berdasarkan review yield historis
# ------------------------------------------------------------

PRIORITY_WINDOWS = [

    # Prioritas 1: Palu, celah setelah
    # window historis 29 September
    (
        "palu_2018",
        "2018-09-30 06:00:00+0700"
    ),

    # Alternatif Palu pada hari yang sama
    (
        "palu_2018",
        "2018-09-30 12:00:00+0700"
    ),

    # Alternatif Lombok untuk keseimbangan kasus
    (
        "lombok_2018",
        "2018-08-11 18:00:00+0700"
    ),

    (
        "palu_2018",
        "2018-09-30 18:00:00+0700"
    ),

    (
        "lombok_2018",
        "2018-08-11 12:00:00+0700"
    ),
]


plan = build_tasks("full")

if len(plan) != 416:
    raise RuntimeError(
        "Struktur FULL plan berubah."
    )

plan_index = {
    (
        t["case"],
        t["query_id"],
        iso_wib(t["start_wib"])
    ): t
    for t in plan
}


def batch7_overlaps(task, state):

    for rec in state.values():

        if rec.get("case") != task["case"]:
            continue

        if rec.get("query_id") != task["query_id"]:
            continue

        if not rec.get("start_wib") or not rec.get("end_wib"):
            raise RuntimeError(
                "Window lama tidak lengkap: "
                + rec["task_id"]
            )

        old_start = datetime.fromisoformat(
            rec["start_wib"]
        )

        old_end = datetime.fromisoformat(
            rec["end_wib"]
        )

        if (
            task["start_wib"] < old_end
            and old_start < task["end_wib"]
        ):
            return True

    return False


candidates = []

for case, start in PRIORITY_WINDOWS:

    key = (case, "E01_P01", start)

    if key not in plan_index:
        raise RuntimeError(
            "Prioritas tidak ada dalam FULL plan: "
            + str(key)
        )

    candidates.append(plan_index[key])


# Fallback dinamis bila daftar prioritas habis.
# Tetap hanya Lombok dan Palu; urut berdasarkan
# kedekatan dengan periode diskusi yang ditentukan.

REFERENCE_DATES = {
    "lombok_2018": datetime(2018, 8, 12).date(),
    "palu_2018": datetime(2018, 10, 1).date(),
}

fallback = [
    t for t in plan
    if (
        t["case"] in REFERENCE_DATES
        and t["query_id"] == "E01_P01"
    )
]

fallback.sort(
    key=lambda t: (
        abs(
            (
                t["start_wib"].date()
                - REFERENCE_DATES[t["case"]]
            ).days
        ),
        0 if t["case"] == "lombok_2018" else 1,
        t["start_wib"],
    )
)

candidates.extend(fallback)


# ------------------------------------------------------------
# D. PILIH SATU TASK AMAN
# ------------------------------------------------------------

selected = None
checked = set()

for candidate in candidates:

    task_id = candidate["task_id"]

    if task_id in checked:
        continue

    checked.add(task_id)

    # Semua task yang sudah ada tidak boleh diulang.
    if task_id in state:
        continue

    # Termasuk overlap dengan run pilot 24 jam.
    if batch7_overlaps(candidate, state):
        continue

    selected = candidate
    break

if selected is None:
    raise RuntimeError(
        "Tidak ada window Lombok/Palu aman yang tersisa "
        "untuk query E01_P01."
    )


# ------------------------------------------------------------
# E. SIMPAN REVIEW DI MEMORY KERNEL
# ------------------------------------------------------------

SELECTED_TASK_ID = selected["task_id"]
REVIEWED_ONE_RUN_TASK_ID = SELECTED_TASK_ID

REVIEWED_TARGET_DETAILS = (
    selected["case"],
    selected["query_id"],
    iso_wib(selected["start_wib"]),
    iso_wib(selected["end_wib"]),
)

REVIEWED_STATE_SHA256 = snapshot[
    "checkpoint_sha"
]

REVIEWED_RAW_MANIFEST_SHA256 = snapshot[
    "raw_manifest_sha"
]

REVIEWED_BASELINE_PROGRESS = (
    CURRENT_RAW_TOTAL,
    CURRENT_UNIQUE_TOTAL,
    len(state)
)

TARGET_CASE = selected["case"]
TARGET_QUERY = selected["query_id"]
TARGET_START = iso_wib(selected["start_wib"])
TARGET_END = iso_wib(selected["end_wib"])


# Periksa bahwa tidak ada perubahan selama preview.
second_snapshot = batch7_scan_saved()

if (
    second_snapshot["checkpoint_sha"]
    != REVIEWED_STATE_SHA256
    or second_snapshot["raw_manifest_sha"]
    != REVIEWED_RAW_MANIFEST_SHA256
):
    raise RuntimeError(
        "Checkpoint/RAW berubah selama preview."
    )


# ------------------------------------------------------------
# F. LAPORAN PREVIEW
# ------------------------------------------------------------

print("=" * 65)
print("BATCH 7 DYNAMIC PREVIEW: PASS")
print("=" * 65)

print("Case       :", TARGET_CASE)
print("Query      :", TARGET_QUERY)
print("Start      :", TARGET_START)
print("End        :", TARGET_END)
print("Task ID    :", SELECTED_TASK_ID)

print("SAVED tasks:", len(state))
print("RAW total  :", CURRENT_RAW_TOTAL)
print("Unique IDs :", CURRENT_UNIQUE_TOTAL)

print("Max new runs: 1")
print("MAX_ITEMS   : 20")
print("Cost cap    : US$0.03")

print("Checkpoint integrity: PASS")
print("RAW integrity       : PASS")
print("Overlap guard       : PASS")
print("Apify calls         : 0")
print("=" * 65)

RAW sekarang: 1379
ID unik sekarang: 372
Kurang RAW: 121
Kurang ID unik: 28
BATCH 7 DYNAMIC PREVIEW: PASS
Case       : lombok_2018
Query      : E01_P01
Start      : 2018-08-11 06:00:00+0700
End        : 2018-08-11 12:00:00+0700
Task ID    : lombok_2018_ead812dc9b218215dbfc
SAVED tasks: 59
RAW total  : 1379
Unique IDs : 372
Max new runs: 1
MAX_ITEMS   : 20
Cost cap    : US$0.03
Checkpoint integrity: PASS
RAW integrity       : PASS
Overlap guard       : PASS
Apify calls         : 0


In [54]:
# 08 — ADAPTIVE SPLIT OPTIONAL; OFF DEFAULT
# Produksi tahap awal tidak memakai recursive split otomatis.
# Ini untuk batch selektif setelah audit, bukan bagian dari full-plan 416 task.

def split_task(task: dict) -> list[dict]:
    if task['depth'] >= MAX_ADAPTIVE_DEPTH:
        return []
    total_seconds = (task['end_wib'] - task['start_wib']).total_seconds()
    if total_seconds > 6 * 3600:
        step = timedelta(hours=6)
    elif total_seconds > 1 * 3600:
        step = timedelta(hours=1)
    elif total_seconds > 15 * 60:
        step = timedelta(minutes=15)
    else:
        return []
    children = []
    start = task['start_wib']
    while start < task['end_wib']:
        stop = min(start + step, task['end_wib'])
        children.append(task_info(
            task['case'], {'query_id': task['query_id'],
                           'base_query': task['base_query']},
            start, stop, depth=task['depth'] + 1))
        start = stop
    return children


## Langkah 5 — Konfirmasi saldo dan batch produksi

`RUN_MODE='full'` adalah **mode perencanaan**, bukan izin otomatis. Sebelum mengaktifkan:

1. Periksa **saldo tersedia aktual** di dashboard Apify; isi `AVAILABLE_CREDIT_USD` dengan nominal *sisa* (bukan $1.21 yang sudah dipakai). Sisakan buffer `$0.50` atau lebih.
2. Pastikan preview menunjukan **416 task awal** (136 Lombok + 136 Palu + 144 Sumatra), **bukan 5.904** atau seluruh 42 query batch. Dengan checkpoint lama, task yang sama akan dilewati.
3. Jalankan cell **tes offline** dan lihat pesan `PASS`. Ada juga **uji fake Actor**, jadi tidak memakai token.
4. Untuk batch pertama, gunakan `MAX_NEW_RUNS = 3`, `MAX_SESSION_SPEND_USD = Decimal('0.12')`, dan `MAX_CHARGE_PER_RUN_USD = Decimal('0.03')`.
5. Baru setelah validasi, isi `PAID_RUN_CONFIRMATION='I_APPROVE_PAID_FULL_SCRAPING'` serta `EXECUTE_SCRAPING=True`, jalankan Cell 04 dan Cell 10. **Cek saldo lagi sesudahnya; jangan gunakan Run All tanpa mengembalikan EXECUTE_SCRAPING=False.**

Guard mencegah panggilan apabila saldo kosong, konfirmasi belum diberikan, atau sisa setelah cadangan lebih kecil dari batas anggaran sesi. Batas per run bukan batas total seluruh sesi; batas sesi dijaga oleh reservasi maksimum biaya setiap run. Apify tetap dapat memiliki pembulatan atau charge tambahan, jadi saldo dashboard adalah sumber utama.

**Jika run gagal di tengah penyimpanan**, checkpoint menjadi `NEEDS_REVIEW`; notebook **tidak** mengulang task itu otomatis. Periksa run dan dataset di Apify sebelum memutuskan pemulihan.


In [55]:
# 09 — PRODUCTION WORKER (SAFE + SELECTED ONE-RUN SUPPORT)
# ============================================================
# Fitur:
# - Konfirmasi eksplisit sebelum scraping berbayar
# - Pemeriksaan saldo dan cadangan kredit
# - Batas maksimum 3 run per sesi
# - Selected one-run test berdasarkan task_id
# - Pencegahan duplicate run
# - Checkpoint append-only: RESERVED -> SAVED / NEEDS_REVIEW
# - Penyimpanan RAW, CSV, dan audit
# - Server-side Apify max_items + max_total_charge_usd
# - Auto-pause setelah 3 run tanpa tweet asli
#
# Tidak menjalankan Actor sampai fungsi run_collection()
# dipanggil dengan gerbang berbayar yang sudah terbuka.
# ============================================================


def decimal_or_none(value):
    if value is None or value == "":
        return None

    try:
        return Decimal(str(value))
    except (ValueError, InvalidOperation):
        return None


# ------------------------------------------------------------
# 09.1 — PAID EXECUTION GATE
# ------------------------------------------------------------

def verify_paid_unlock():
    if not EXECUTE_SCRAPING:
        raise RuntimeError(
            "Scraping terkunci: EXECUTE_SCRAPING=False."
        )

    if PAID_RUN_CONFIRMATION != (
        "I_APPROVE_PAID_FULL_SCRAPING"
    ):
        raise RuntimeError(
            "Konfirmasi scraping berbayar belum sesuai."
        )

    available = decimal_or_none(
        AVAILABLE_CREDIT_USD
    )

    if available is None or available <= 0:
        raise RuntimeError(
            "Isi AVAILABLE_CREDIT_USD sesuai saldo "
            "terbaru dari dashboard Apify."
        )

    if AVAILABLE_CREDIT_USD is None:
        raise RuntimeError(
            "Saldo belum dikonfirmasi."
        )

    if available - CREDIT_RESERVE_USD < (
        MAX_SESSION_SPEND_USD
    ):
        raise RuntimeError(
            "Saldo tidak cukup untuk budget sesi "
            "ditambah cadangan kredit. "
            f"Saldo={available}, "
            f"Reserve={CREDIT_RESERVE_USD}, "
            f"Session={MAX_SESSION_SPEND_USD}"
        )

    if ENABLE_ADAPTIVE:
        raise RuntimeError(
            "Adaptive scraping harus OFF."
        )

    if RETRY_UNCERTAIN_TASKS:
        raise RuntimeError(
            "Retry otomatis untuk task ambigu "
            "tidak diizinkan."
        )

    if MAX_CHARGE_PER_RUN_USD <= 0:
        raise RuntimeError(
            "MAX_CHARGE_PER_RUN_USD tidak valid."
        )

    if MAX_SESSION_SPEND_USD < (
        MAX_CHARGE_PER_RUN_USD
    ):
        raise RuntimeError(
            "Budget sesi lebih kecil dari cap per run."
        )

    return True


# ------------------------------------------------------------
# 09.2 — COLLECT ONE TASK
# ------------------------------------------------------------

def collect_one(
    client,
    task: dict,
    per_run_cap_usd: Decimal
) -> dict:

    run_input = {
        "searchTerms": [task["query"]],
        "maxItems": MAX_ITEMS,
        "queryType": "Latest",
    }

    started = time.monotonic()

    task = {
        **task,
        "per_run_cap_usd": str(per_run_cap_usd)
    }

    # Batas diterapkan pada API Apify.
    run_result = client.actor(ACTOR_ID).call(
        run_input=run_input,
        max_items=MAX_ITEMS,
        max_total_charge_usd=per_run_cap_usd,
        restart_on_error=False,
    )

    if run_result is None:
        raise RuntimeError(
            "Apify tidak mengembalikan run result. "
            "Periksa dashboard sebelum retry."
        )

    run_dict = run_to_dict(run_result)

    if run_dict.get("status") != "SUCCEEDED":
        raise RuntimeError(
            "Actor tidak berstatus SUCCEEDED: "
            f"{run_dict.get('status')}"
        )

    dataset_id = (
        run_dict.get("defaultDatasetId")
        or run_dict.get("default_dataset_id")
    )

    if not dataset_id:
        raise RuntimeError(
            "Dataset ID tidak ditemukan. "
            "Jangan retry tanpa pemeriksaan manual."
        )

    items = list(
        client.dataset(dataset_id).iterate_items()
    )

    if not all(isinstance(item, dict) for item in items):
        raise TypeError(
            "Dataset mengandung item non-dictionary."
        )

    # -------------------------------------------
    # Simpan RAW terlebih dahulu.
    # Mock dan duplikat tidak dibuang dari RAW.
    # -------------------------------------------

    raw_file = (
        RAW_DIR
        / task["case"]
        / f"{task['task_id']}.jsonl"
    )

    json_write_atomic(raw_file, items)

    if items:
        csv_file = (
            CSV_DIR
            / task["case"]
            / f"{task['task_id']}.csv"
        )

        csv_write_atomic(
            csv_file,
            pd.DataFrame(items)
        )

    # -------------------------------------------
    # Audit kualitas data
    # -------------------------------------------

    df_audit, metrics = audit_items(
        items,
        task
    )

    flags = quality_flags(metrics)

    audit_file = (
        AUDIT_DIR
        / task["case"]
        / f"{task['task_id']}_audit.csv"
    )

    csv_write_atomic(
        audit_file,
        df_audit
    )

    # -------------------------------------------
    # Simpan checkpoint SAVED.
    # -------------------------------------------

    record = make_run_record(
        task,
        run_dict,
        metrics,
        flags,
        raw_file,
        audit_file
    )

    record["elapsed_seconds"] = round(
        time.monotonic() - started,
        2
    )

    append_checkpoint(record)

    return record


# ------------------------------------------------------------
# 09.3 — COLLECTION WORKER
# ------------------------------------------------------------

def run_collection(
    client,
    mode=RUN_MODE,
    max_new_runs=MAX_NEW_RUNS,
    selected_task_id=None
):
    """
    Menjalankan pengumpulan data dengan dua mode:

    A. Regular collection
       max_new_runs=3
       selected_task_id=None

    B. Selected one-run test
       max_new_runs=1
       selected_task_id=<task_id>

    Semua task yang sudah tercatat dalam checkpoint
    tidak akan dieksekusi ulang secara otomatis.
    """

    # -------------------------------------------
    # VALIDASI PARAMETER SESI
    # -------------------------------------------

    if (
        type(max_new_runs) is not int
        or not 1 <= max_new_runs <= 3
    ):
        raise RuntimeError(
            "max_new_runs harus integer 1 sampai 3."
        )

    if max_new_runs > MAX_NEW_RUNS:
        raise RuntimeError(
            "Jumlah run melebihi MAX_NEW_RUNS "
            "yang dikonfigurasi pada Cell 04."
        )

    if selected_task_id is not None:
        if mode != "full":
            raise RuntimeError(
                "Selected one-run hanya berlaku "
                "untuk RUN_MODE='full'."
            )

        if max_new_runs != 1:
            raise RuntimeError(
                "Selected task hanya boleh "
                "dijalankan dengan max_new_runs=1."
            )

        if not isinstance(selected_task_id, str):
            raise TypeError(
                "selected_task_id harus berupa string."
            )

        if not selected_task_id.strip():
            raise ValueError(
                "selected_task_id tidak boleh kosong."
            )

        # Wajib sama dengan target yang sudah
        # lulus preview offline Cell 07D.
        if globals().get(
            "REVIEWED_ONE_RUN_TASK_ID"
        ) != selected_task_id:
            raise RuntimeError(
                "Selected task belum lolos "
                "preview offline Cell 07D."
            )

    elif max_new_runs == 1:
        raise RuntimeError(
            "One-run test wajib menggunakan "
            "selected_task_id."
        )

    # -------------------------------------------
    # BUILD FULL PLAN + PREFLIGHT
    # -------------------------------------------

    tasks = build_tasks(mode)

    preflight(tasks, mode)

    verify_paid_unlock()

    state = load_checkpoint()

    # -------------------------------------------
    # SELECTED TASK ROUTING
    # -------------------------------------------

    if selected_task_id is not None:

        matches = [
            task
            for task in tasks
            if task["task_id"] == selected_task_id
        ]

        if len(matches) != 1:
            raise RuntimeError(
                "Selected task tidak ditemukan "
                "secara unik dalam FULL plan."
            )

        if selected_task_id in state:
            raise RuntimeError(
                "Selected task sudah tercatat "
                "di checkpoint. "
                "Eksekusi dibatalkan untuk "
                "mencegah duplikasi biaya."
            )

        # Hanya satu task yang boleh masuk worker.
        tasks = matches

        print("\nSELECTED ONE-RUN MODE")
        print("Task ID:", selected_task_id)

    else:
        print("\nREGULAR COLLECTION MODE")
        print("Maximum runs:", max_new_runs)

    # -------------------------------------------
    # VALIDASI BUDGET SESI
    # -------------------------------------------

    required_upper = (
        MAX_CHARGE_PER_RUN_USD * max_new_runs
    )

    if required_upper > MAX_SESSION_SPEND_USD:
        raise RuntimeError(
            "Budget sesi tidak mencukupi "
            "untuk jumlah run yang diminta. "
            f"Required={required_upper}, "
            f"Budget={MAX_SESSION_SPEND_USD}"
        )

    reserved_upper_usd = Decimal("0")

    used = 0
    consecutive_no_real = 0

    saved = []
    errors = []
    blocked = 0

    # -------------------------------------------
    # MAIN COLLECTION LOOP
    # -------------------------------------------

    for task in tasks:

        if used >= max_new_runs:
            break

        task_id = task["task_id"]

        existing = state.get(task_id)

        # ---------------------------------------
        # CHECKPOINT DUPLICATE GUARD
        # ---------------------------------------

        if existing is not None:

            status = existing.get("status")

            if status == "SAVED":
                print(
                    "SKIP SAVED:",
                    task_id
                )
                continue

            if status in {
                "RESERVED",
                "NEEDS_REVIEW",
                "FAILED",
                "FAILED_NEEDS_REVIEW"
            }:
                blocked += 1

                print(
                    "SKIP AMBIGUOUS:",
                    task_id,
                    "|",
                    status
                )
                continue

            raise RuntimeError(
                "Status checkpoint tidak dikenal: "
                f"{task_id} -> {status}"
            )

        # ---------------------------------------
        # BUDGET RESERVATION
        # ---------------------------------------

        remaining = (
            MAX_SESSION_SPEND_USD
            - reserved_upper_usd
        )

        run_cap = min(
            MAX_CHARGE_PER_RUN_USD,
            remaining
        )

        if run_cap < MAX_CHARGE_PER_RUN_USD:
            print(
                "STOP: budget sesi tidak mencukupi."
            )
            break

        if run_cap <= 0:
            raise RuntimeError(
                "Run cap harus lebih dari nol."
            )

        used += 1

        reserved_upper_usd += run_cap

        print(
            f"\n[{used}/{max_new_runs}] "
            f"{task['case']} "
            f"{task['query_id']}"
        )

        print(
            "  Window:",
            iso_wib(task["start_wib"]),
            "→",
            iso_wib(task["end_wib"])
        )

        print(
            "  Task ID:",
            task_id
        )

        print(
            "  Cap maksimum run:",
            f"${run_cap}"
        )

        print(
            "  Cap kumulatif sesi:",
            f"${reserved_upper_usd}"
        )

        # ---------------------------------------
        # RESERVED — sebelum panggilan berbayar
        # ---------------------------------------

        start_record = {
            "task_id": task_id,
            "status": "RESERVED",
            "case": task["case"],
            "query_id": task["query_id"],
            "query": task["query"],
            "actor_id": ACTOR_ID,
            "max_total_charge_usd_requested": str(
                run_cap
            ),
            "timestamp_utc": datetime.now(
                UTC
            ).isoformat()
        }

        append_checkpoint(start_record)

        state[task_id] = start_record

        # ---------------------------------------
        # API CALL
        # ---------------------------------------

        try:

            record = collect_one(
                client,
                task,
                per_run_cap_usd=run_cap
            )

            state[task_id] = record

            saved.append(record)

            print(
                "  Raw:",
                record["rows_raw"],
                "| ID unik dalam run:",
                record["unique_real_ids"],
                "| Mock:",
                record["rows_mock"]
            )

            print(
                "  Quality:",
                ", ".join(
                    record["quality_flags"]
                )
            )

            print(
                "  Data quality:",
                record["data_quality_status"]
            )

            print(
                "  Kandidat policy:",
                record["unique_policy_candidate_ids"]
            )

            # -----------------------------------
            # AUTO-PAUSE MONITOR
            # -----------------------------------

            if record["rows_real_tweet"] == 0:
                consecutive_no_real += 1
            else:
                consecutive_no_real = 0

            print(
                "  Consecutive no-real:",
                consecutive_no_real
            )

            # -----------------------------------
            # COST MONITOR
            # -----------------------------------

            cost = decimal_or_none(
                record.get("usage_total_usd")
            )

            if cost is not None:

                print(
                    "  Usage reported (USD):",
                    cost
                )

                if cost > run_cap:
                    print(
                        "  WARNING: usage melebihi "
                        "cap yang diminta. "
                        "Periksa dashboard Apify."
                    )

            else:
                print(
                    "  Usage belum tersedia. "
                    "Periksa dashboard Apify."
                )

            # -----------------------------------
            # AUTO-PAUSE
            # -----------------------------------

            if (
                consecutive_no_real
                >= MAX_CONSECUTIVE_NO_REAL
            ):

                print(
                    "\nAUTO-PAUSE: terlalu banyak "
                    "run tanpa tweet asli."
                )

                print(
                    "Evaluasi query, tanggal, "
                    "dan perilaku Actor."
                )

                break

        # ---------------------------------------
        # FAIL-CLOSED ERROR HANDLING
        # ---------------------------------------

        except Exception as exc:

            entry = {
                "task_id": task_id,
                "status": "NEEDS_REVIEW",
                "case": task["case"],
                "query_id": task["query_id"],
                "query": task["query"],
                "max_total_charge_usd_requested": str(
                    run_cap
                ),
                "timestamp_utc": datetime.now(
                    UTC
                ).isoformat(),
                "error": (
                    f"{type(exc).__name__}: {exc}"
                )
            }

            append_checkpoint(entry)

            state[task_id] = entry

            errors.append(entry)

            print(
                "  STOP / NEEDS_REVIEW:",
                entry["error"]
            )

            print(
                "  Jangan retry otomatis. "
                "Periksa Actor run di Apify."
            )

            break

        finally:
            if (
                REQUEST_DELAY_SECONDS > 0
                and used < max_new_runs
            ):
                time.sleep(
                    REQUEST_DELAY_SECONDS
                )

    # -------------------------------------------
    # FINAL SESSION SUMMARY
    # -------------------------------------------

    print("\n" + "=" * 65)
    print("COLLECTION SESSION COMPLETE")

    print(
        "Actor calls baru:",
        used
    )

    print(
        "SAVED baru:",
        len(saved)
    )

    print(
        "NEEDS_REVIEW baru:",
        len(errors)
    )

    print(
        "Task ambigu lama dilewati:",
        blocked
    )

    print(
        "Cap komitmen sesi:",
        f"${reserved_upper_usd}"
    )

    print(
        "Periksa saldo aktual Apify "
        "sebelum memulai sesi berikutnya."
    )

    print("=" * 65)

    return saved, errors


In [56]:
# 09B — TEST REGRESI TANPA AKSES INTERNET / TANPA TOKEN APIFY
# Tes ini tidak mengubah checkpoint penelitian sungguhan.
from tempfile import TemporaryDirectory

_plan = build_tasks('full')
assert preflight(_plan, 'full')
assert len(_plan) == 416, f'Task full bukan 416: {len(_plan)}'
assert {c: sum(t['case'] == c for t in _plan) for c in FULL_CASES} == {
    'lombok_2018': 136, 'palu_2018': 136, 'sumatra_2025': 144,
}
assert all(t['query_id'] == 'E01_P01' for t in _plan)
assert len({t['task_id'] for t in _plan}) == len(_plan)
assert all(t['end_wib'] - t['start_wib'] == timedelta(hours=6) for t in _plan)

# Cross-check ke checkpoint yang sudah terbukti dari pilot (P5 dan P8part2).
_sconf = DISASTER_CASES['sumatra_2025']
_batch = _sconf['query_batches'][0]
assert task_info('sumatra_2025', _batch,
                wib_dt('2025-12-03', 6), wib_dt('2025-12-03', 12))['task_id'] == (
    'sumatra_2025_4f7c9d0f6659448b9b30'
)
assert task_info('sumatra_2025', _batch,
                wib_dt('2025-12-03', 6), wib_dt('2025-12-03', 9))['task_id'] == (
    'sumatra_2025_92a73c19df027e14390c'
)

# Verifikasi benar-benar skip checkpoint lama dalam rencana full (khusus exact task).
_old_state = load_checkpoint()
_old_saved_count = sum(_old_state.get(t['task_id'], {}).get('status') == 'SAVED'
                    for t in _plan)
print('Task yang cocok checkpoint pilot dan akan dilewati:', _old_saved_count)

# Simulasi Actor OFFLINE. Tidak ada panggilan dari apify_client.
class _FakeActor:
    def __init__(self, parent):
        self.parent = parent
    def call(self, **kwargs):
        self.parent.calls.append(kwargs)
        return {'id': 'fake-offline-run', 'status': 'SUCCEEDED',
                'defaultDatasetId': 'fake-offline-ds', 'usageTotalUsd': 0.001}

class _FakeDataset:
    def __init__(self, items):
        self.items = items
    def iterate_items(self):
        return iter(self.items)

class _FakeClient:
    def __init__(self, items):
        self.items = items
        self.calls = []
    def actor(self, actor_id):
        assert actor_id == ACTOR_ID
        return _FakeActor(self)
    def dataset(self, dataset_id):
        assert dataset_id == 'fake-offline-ds'
        return _FakeDataset(self.items)

_saved_globals = {k: globals()[k] for k in ('PROJECT_ROOT', 'RAW_DIR', 'CSV_DIR',
                                            'AUDIT_DIR', 'CHECKPOINT_FILE')}
try:
    with TemporaryDirectory() as _td:
        _root = Path(_td)
        PROJECT_ROOT = _root
        RAW_DIR = _root / '01_Raw Data' / 'outputs' / 'raw_jsonl'
        CSV_DIR = _root / '01_Raw Data' / 'outputs' / 'raw_csv'
        AUDIT_DIR = _root / '01_Raw Data' / 'outputs' / 'audit'
        CHECKPOINT_FILE = _root / '01_Raw Data' / 'outputs' / 'state' / 'collection_checkpoint.jsonl'
        _fake_task = _plan[0]
        _fake_dt = (_fake_task['start_wib'] + timedelta(minutes=1)).astimezone(UTC)
        _fake_event = DISASTER_CASES[_fake_task['case']]['event_terms'][0]
        _fake_tweet = {'type': 'tweet', 'id': 'offline-test-1',
                    'createdAt': _fake_dt.strftime('%a %b %d %H:%M:%S %z %Y'),
                    'text': f'{_fake_event} semestinya bencana nasional', 'lang': 'id'}
        _fake_client = _FakeClient([_fake_tweet, dict(_fake_tweet),
                                    {'type': 'mock_tweet', 'id': -1, 'text': 'mock'}])
        _fakerecord = collect_one(_fake_client, _fake_task, Decimal('0.03'))
        assert _fakerecord['rows_raw'] == 3
        assert _fakerecord['rows_mock'] == 1
        assert _fakerecord['unique_real_ids'] == 1
        assert _fakerecord['duplicate_real_ids_in_dataset'] == 1
        assert load_checkpoint()[_fake_task['task_id']]['status'] == 'SAVED'
        _call = _fake_client.calls[0]
        assert _call['max_items'] == 20
        assert _call['max_total_charge_usd'] == Decimal('0.03')
        assert _call['run_input']['maxItems'] == 20
        assert _call['restart_on_error'] is False
finally:
    globals().update(_saved_globals)

print('UNIT TEST OFFLINE: PASS — 416 planned, hash kompatibel, fake Actor + checkpoint aman')


Task yang cocok checkpoint pilot dan akan dilewati: 47
UNIT TEST OFFLINE: PASS — 416 planned, hash kompatibel, fake Actor + checkpoint aman


In [57]:
# ONE-RUN ROUTING REGRESSION — OFFLINE ONLY
from tempfile import TemporaryDirectory

assert EXECUTE_SCRAPING is False
assert REVIEWED_ONE_RUN_TASK_ID == SELECTED_TASK_ID

saved_globals = {
    k: globals()[k]
    for k in (
        "PROJECT_ROOT", "RAW_DIR", "CSV_DIR",
        "AUDIT_DIR", "CHECKPOINT_FILE",
        "EXECUTE_SCRAPING", "PAID_RUN_CONFIRMATION"
    )
}

real_checkpoint_hash = hashlib.sha256(
    CHECKPOINT_FILE.read_bytes()
).hexdigest()

try:
    with TemporaryDirectory() as tmp:
        root = Path(tmp)

        PROJECT_ROOT = root
        RAW_DIR = root / "raw_jsonl"
        CSV_DIR = root / "raw_csv"
        AUDIT_DIR = root / "audit"
        CHECKPOINT_FILE = root / "checkpoint.jsonl"

        # Izin sementara HANYA untuk FakeClient.
        EXECUTE_SCRAPING = True
        PAID_RUN_CONFIRMATION = (
            "I_APPROVE_PAID_FULL_SCRAPING"
        )

        fake = _FakeClient([
            {
                "type": "mock_tweet",
                "id": -1,
                "text": "Offline one-run test"
            }
        ])

        saved, errors = run_collection(
            fake,
            mode="full",
            max_new_runs=1,
            selected_task_id=SELECTED_TASK_ID
        )

        assert len(fake.calls) == 1
        assert len(saved) == 1
        assert len(errors) == 0
        assert saved[0]["task_id"] == SELECTED_TASK_ID

        # Ulang task yang sama harus ditolak.
        try:
            run_collection(
                fake,
                mode="full",
                max_new_runs=1,
                selected_task_id=SELECTED_TASK_ID
            )
        except RuntimeError:
            pass
        else:
            raise AssertionError(
                "Duplicate-run guard gagal."
            )

        assert len(fake.calls) == 1

finally:
    globals().update(saved_globals)

assert hashlib.sha256(
    CHECKPOINT_FILE.read_bytes()
).hexdigest() == real_checkpoint_hash

print("ONE-RUN ROUTING REGRESSION: PASS")
print("Selected task:", SELECTED_TASK_ID)
print("Fake actor calls: 1")
print("Duplicate-run guard: PASS")
print("Research checkpoint unchanged: TRUE")
print("Real Apify calls: 0")



SELECTED ONE-RUN MODE
Task ID: lombok_2018_ead812dc9b218215dbfc

[1/1] lombok_2018 E01_P01
  Window: 2018-08-11 06:00:00+0700 → 2018-08-11 12:00:00+0700
  Task ID: lombok_2018_ead812dc9b218215dbfc
  Cap maksimum run: $0.03
  Cap kumulatif sesi: $0.03
  Raw: 1 | ID unik dalam run: 0 | Mock: 1
  Quality: MOCK_PRESENT, NO_REAL_TWEET
  Data quality: NO_REAL_DATA
  Kandidat policy: 0
  Consecutive no-real: 1
  Usage reported (USD): 0.001

COLLECTION SESSION COMPLETE
Actor calls baru: 1
SAVED baru: 1
NEEDS_REVIEW baru: 0
Task ambigu lama dilewati: 0
Cap komitmen sesi: $0.03
Periksa saldo aktual Apify sebelum memulai sesi berikutnya.
ONE-RUN ROUTING REGRESSION: PASS
Selected task: lombok_2018_ead812dc9b218215dbfc
Fake actor calls: 1
Duplicate-run guard: PASS
Research checkpoint unchanged: TRUE
Real Apify calls: 0


In [ ]:
# 09C — AUTO-PAUSE REGRESSION TEST (OFFLINE)
# ============================================================
# FakeClient only — TIDAK MEMANGGIL APIFY.
# Maksimum 3 run dan auto-pause setelah 3 NO_REAL_DATA.
# Checkpoint penelitian tidak diubah.
# ============================================================

from tempfile import TemporaryDirectory
from pathlib import Path
import hashlib

assert EXECUTE_SCRAPING is False
assert PAID_RUN_CONFIRMATION == ""
assert RUN_MODE == "full"
assert MAX_NEW_RUNS == 3
assert MAX_SESSION_SPEND_USD >= Decimal("0.09")
assert ENABLE_ADAPTIVE is False
assert "_FakeClient" in globals()

real_checkpoint = CHECKPOINT_FILE

hash_before = hashlib.sha256(
    real_checkpoint.read_bytes()
).hexdigest()

saved_globals = {
    name: globals()[name]
    for name in (
        "PROJECT_ROOT",
        "RAW_DIR",
        "CSV_DIR",
        "AUDIT_DIR",
        "CHECKPOINT_FILE",
        "EXECUTE_SCRAPING",
        "PAID_RUN_CONFIRMATION",
    )
}

try:
    with TemporaryDirectory() as tmp:

        root = Path(tmp)

        PROJECT_ROOT = root
        RAW_DIR = root / "raw_jsonl"
        CSV_DIR = root / "raw_csv"
        AUDIT_DIR = root / "audit"
        CHECKPOINT_FILE = root / "checkpoint.jsonl"

        # Hanya membuka gerbang untuk FakeClient.
        EXECUTE_SCRAPING = True
        PAID_RUN_CONFIRMATION = (
            "I_APPROVE_PAID_FULL_SCRAPING"
        )

        fake_client = _FakeClient([
            {
                "type": "mock_tweet",
                "id": -1,
                "text": "Offline simulated empty result"
            }
        ])

        saved, errors = run_collection(
            fake_client,
            mode="full",
            max_new_runs=3,
            selected_task_id=None
        )

        # Verifikasi batas maksimal run.
        assert len(fake_client.calls) == 3
        assert len(saved) == 3
        assert len(errors) == 0

        # Semua run menghasilkan NO_REAL_DATA.
        assert all(
            record["data_quality_status"]
            == "NO_REAL_DATA"
            for record in saved
        )

        assert all(
            record["rows_real_tweet"] == 0
            for record in saved
        )

        # Maksimum 3 task yang tersimpan di sandbox.
        fake_state = load_checkpoint()

        assert len(fake_state) == 3
        assert all(
            record["status"] == "SAVED"
            for record in fake_state.values()
        )

        # Batas biaya harus benar-benar diteruskan
        # melalui pemanggilan ActorClient.
        assert all(
            call["max_total_charge_usd"]
            == Decimal("0.03")
            for call in fake_client.calls
        )

        assert all(
            call["restart_on_error"] is False
            for call in fake_client.calls
        )

finally:
    # Pulihkan semua variabel penelitian.
    globals().update(saved_globals)

hash_after = hashlib.sha256(
    real_checkpoint.read_bytes()
).hexdigest()

assert hash_before == hash_after
assert len(load_checkpoint()) == 42

print("=" * 60)
print("AUTO-PAUSE OFFLINE: PASS")
print("Fake Actor calls: 3")
print("NO_REAL_DATA runs: 3")
print("Maximum run limit: PASS")
print("Checkpoint unchanged: TRUE")
print("Real Apify calls: 0")
print("=" * 60)


In [59]:
# 09D — BATCH 7 DYNAMIC ONE-RUN PREFLIGHT
# OFFLINE + PAID VERIFICATION
# ============================================================

from decimal import Decimal
from datetime import datetime


def preflight_selected_one_run(
    require_paid_unlock=False
):

    # --------------------------------------------------------
    # 1. KONFIGURASI SATU RUN
    # --------------------------------------------------------

    if RUN_MODE != "full":
        raise RuntimeError(
            "RUN_MODE harus full."
        )

    if MAX_NEW_RUNS != 1:
        raise RuntimeError(
            "MAX_NEW_RUNS harus 1."
        )

    if MAX_ITEMS != 20:
        raise RuntimeError(
            "MAX_ITEMS harus 20."
        )

    if ENABLE_ADAPTIVE or RETRY_UNCERTAIN_TASKS:
        raise RuntimeError(
            "Adaptive dan retry otomatis harus OFF."
        )

    if (
        Decimal(str(MAX_SESSION_SPEND_USD))
        != Decimal("0.03")
    ):
        raise RuntimeError(
            "Budget sesi harus US$0.03."
        )

    if (
        Decimal(str(MAX_CHARGE_PER_RUN_USD))
        != Decimal("0.03")
    ):
        raise RuntimeError(
            "Cap per run harus US$0.03."
        )

    if (
        Decimal(str(CREDIT_RESERVE_USD))
        < Decimal("0.50")
    ):
        raise RuntimeError(
            "Cadangan kredit kurang dari US$0.50."
        )


    # --------------------------------------------------------
    # 2. VERIFIKASI TARGET CELL 07D
    # --------------------------------------------------------

    task_id = globals().get(
        "REVIEWED_ONE_RUN_TASK_ID"
    )

    if not task_id:
        raise RuntimeError(
            "Cell 07D belum dijalankan."
        )

    if task_id != globals().get("SELECTED_TASK_ID"):
        raise RuntimeError(
            "Selected task berubah."
        )

    reviewed_details = globals().get(
        "REVIEWED_TARGET_DETAILS"
    )

    if not reviewed_details:
        raise RuntimeError(
            "Detail target belum direview."
        )

    if reviewed_details[0] not in (
        "lombok_2018",
        "palu_2018"
    ):
        raise RuntimeError(
            "Batch 7 dibatasi untuk Lombok/Palu."
        )

    if reviewed_details[1] != "E01_P01":
        raise RuntimeError(
            "Query Batch 7 tidak sesuai."
        )


    # --------------------------------------------------------
    # 3. AUDIT ULANG CHECKPOINT DAN RAW
    # --------------------------------------------------------

    if "batch7_scan_saved" not in globals():
        raise RuntimeError(
            "Fungsi audit Cell 07D belum tersedia."
        )

    snapshot = batch7_scan_saved()
    state = snapshot["state"]

    if snapshot["checkpoint_sha"] != globals().get(
        "REVIEWED_STATE_SHA256"
    ):
        raise RuntimeError(
            "Checkpoint berubah sejak preview. "
            "Ulangi review OFFLINE sebelum paid run."
        )

    if snapshot["raw_manifest_sha"] != globals().get(
        "REVIEWED_RAW_MANIFEST_SHA256"
    ):
        raise RuntimeError(
            "RAW berubah sejak preview."
        )

    progress_now = (
        snapshot["raw_total"],
        snapshot["unique_total"],
        len(state)
    )

    if progress_now != globals().get(
        "REVIEWED_BASELINE_PROGRESS"
    ):
        raise RuntimeError(
            "Jumlah data berubah sejak preview."
        )

    if task_id in state:
        raise RuntimeError(
            "Task sudah tercatat. Dilarang retry."
        )


    # --------------------------------------------------------
    # 4. VALIDASI TARGET DALAM FULL PLAN
    # --------------------------------------------------------

    plan = build_tasks("full")

    if len(plan) != 416:
        raise RuntimeError(
            "FULL plan berubah."
        )

    matches = [
        t for t in plan
        if t["task_id"] == task_id
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "Task ID tidak unik."
        )

    selected = matches[0]

    details = (
        selected["case"],
        selected["query_id"],
        iso_wib(selected["start_wib"]),
        iso_wib(selected["end_wib"]),
    )

    if details != reviewed_details:
        raise RuntimeError(
            "Detail task berubah sejak Cell 07D."
        )

    if (
        selected["end_wib"] - selected["start_wib"]
    ) != timedelta(hours=6):
        raise RuntimeError(
            "Window harus 6 jam."
        )


    # --------------------------------------------------------
    # 5. PERIKSA OVERLAP SEMUA TASK LAMA
    # --------------------------------------------------------

    if batch7_overlaps(selected, state):
        raise RuntimeError(
            "Window overlap dengan task sebelumnya."
        )


    # --------------------------------------------------------
    # 6. TARGET PENGUMPULAN
    # --------------------------------------------------------

    if (
        snapshot["raw_total"] >= 1500
        and snapshot["unique_total"] >= 400
    ):
        raise RuntimeError(
            "Target 1500 RAW dan 400 ID telah tercapai. "
            "STOP paid scraping."
        )


    # --------------------------------------------------------
    # 7. MODE OFFLINE / PAID
    # --------------------------------------------------------

    if require_paid_unlock:

        # Fungsi Cell 09: cek konfirmasi, saldo,
        # cadangan, dan budget sebelum Actor call.
        verify_paid_unlock()

    else:

        if EXECUTE_SCRAPING is not False:
            raise RuntimeError(
                "OFFLINE preflight membutuhkan "
                "EXECUTE_SCRAPING=False."
            )

        if PAID_RUN_CONFIRMATION != "":
            raise RuntimeError(
                "Konfirmasi berbayar harus kosong."
            )


    # --------------------------------------------------------
    # 8. PASTIKAN DATA TIDAK BERUBAH SELAMA PREFLIGHT
    # --------------------------------------------------------

    final_snapshot = batch7_scan_saved()

    if (
        final_snapshot["checkpoint_sha"]
        != snapshot["checkpoint_sha"]
        or final_snapshot["raw_manifest_sha"]
        != snapshot["raw_manifest_sha"]
    ):
        raise RuntimeError(
            "File berubah selama preflight. STOP."
        )


    # --------------------------------------------------------
    # 9. HASIL
    # --------------------------------------------------------

    print("=" * 65)
    print("BATCH 7 ONE-RUN PREFLIGHT: PASS")
    print("=" * 65)

    print(
        "Mode:",
        "PAID VERIFIED"
        if require_paid_unlock
        else "OFFLINE"
    )

    print("Case      :", selected["case"])
    print("Query     :", selected["query_id"])
    print("Start     :", details[2])
    print("End       :", details[3])
    print("Task ID   :", task_id)

    print("SAVED tasks:", len(state))
    print("RAW total  :", snapshot["raw_total"])
    print("Unique IDs :", snapshot["unique_total"])

    print("Max Actor calls : 1")
    print("Requested cap   : US$0.03")

    print("Checkpoint guard: PASS")
    print("RAW guard       : PASS")
    print("Overlap guard   : PASS")
    print("=" * 65)

    return selected


# ------------------------------------------------------------
# 10. AUTO-PREFLIGHT HANYA DALAM MODE OFFLINE
# ------------------------------------------------------------

if EXECUTE_SCRAPING is False:

    PREFLIGHT_SELECTED_TASK = (
        preflight_selected_one_run(
            require_paid_unlock=False
        )
    )

else:

    print(
        "PAID preflight akan dijalankan "
        "oleh Cell 10."
    )


BATCH 7 ONE-RUN PREFLIGHT: PASS
Mode: OFFLINE
Case      : lombok_2018
Query     : E01_P01
Start     : 2018-08-11 06:00:00+0700
End       : 2018-08-11 12:00:00+0700
Task ID   : lombok_2018_ead812dc9b218215dbfc
SAVED tasks: 59
RAW total  : 1379
Unique IDs : 372
Max Actor calls : 1
Requested cap   : US$0.03
Checkpoint guard: PASS
RAW guard       : PASS
Overlap guard   : PASS


In [74]:
# 10XGate — SELECTED ONE-RUN EXECUTION GATE
# ============================================================
# DEFAULT: OFFLINE / PREVIEW ONLY
#
# Target ditentukan secara dinamis oleh Cell 07D.
# Hanya menjalankan satu task yang lolos Cell 09D.
# Tidak memakai antrean default tiga kasus.
# ============================================================

print("=" * 65)
print("SELECTED ONE-RUN EXECUTION GATE")

print("RUN_MODE:", RUN_MODE)
print("EXECUTE_SCRAPING:", EXECUTE_SCRAPING)
print("MAX_NEW_RUNS:", MAX_NEW_RUNS)

if EXECUTE_SCRAPING is False:

    # --------------------------------------------------------
    # OFFLINE MODE
    # --------------------------------------------------------

    assert PAID_RUN_CONFIRMATION == ""

    selected = preflight_selected_one_run(
        require_paid_unlock=False
    )

    print("\nPREVIEW ONLY — NO API CALL")
    print("Selected task:", selected["task_id"])
    print("Existing checkpoint remains unchanged.")
    print("Paid scraping is LOCKED.")

elif EXECUTE_SCRAPING is True:

    # --------------------------------------------------------
    # PAID MODE — ONE SELECTED TASK ONLY
    # --------------------------------------------------------

    try:

        # Periksa ulang seluruh kondisi SEBELUM
        # mengambil token atau membuat Actor run.
        selected = preflight_selected_one_run(
            require_paid_unlock=True
        )

        print("\nPAID ONE-RUN MODE")
        print("Target:", selected["task_id"])
        print("Maximum Actor calls: 1")

        # Token hanya diminta setelah preflight PASS.
        _client = get_apify_client()

        # Kirim satu task yang sudah diverifikasi.
        _saved, _errors = run_collection(
            _client,
            mode="full",
            max_new_runs=1,
            selected_task_id=selected["task_id"]
        )

        if _errors:
            print(
                "\nWARNING: task memerlukan review."
            )
            print(
                "Jangan retry sebelum memeriksa Apify."
            )

        else:
            print(
                "\nOne-run session finished."
            )

        print(
            "Saved runs:",
            len(_saved)
        )

        print(
            "Review-needed runs:",
            len(_errors)
        )

    finally:

        # Kunci kembali runtime setelah percobaan,
        # termasuk apabila terjadi error.
        EXECUTE_SCRAPING = False
        PAID_RUN_CONFIRMATION = ""

        print(
            "\nRUNTIME PAID GATE LOCKED."
        )

        print(
            "Pastikan Cell 04 juga dikembalikan "
            "ke konfigurasi OFF sebelum menyimpan."
        )

else:

    raise RuntimeError(
        "EXECUTE_SCRAPING harus boolean True/False."
    )

print("=" * 65)


SELECTED ONE-RUN EXECUTION GATE
RUN_MODE: full
EXECUTE_SCRAPING: True
MAX_NEW_RUNS: 1
BATCH 7 ONE-RUN PREFLIGHT: PASS
Mode: PAID VERIFIED
Case      : lombok_2018
Query     : E01_P01
Start     : 2018-08-11 06:00:00+0700
End       : 2018-08-11 12:00:00+0700
Task ID   : lombok_2018_ead812dc9b218215dbfc
SAVED tasks: 59
RAW total  : 1379
Unique IDs : 372
Max Actor calls : 1
Requested cap   : US$0.03
Checkpoint guard: PASS
RAW guard       : PASS
Overlap guard   : PASS

PAID ONE-RUN MODE
Target: lombok_2018_ead812dc9b218215dbfc
Maximum Actor calls: 1

SELECTED ONE-RUN MODE
Task ID: lombok_2018_ead812dc9b218215dbfc

[1/1] lombok_2018 E01_P01
  Window: 2018-08-11 06:00:00+0700 → 2018-08-11 12:00:00+0700
  Task ID: lombok_2018_ead812dc9b218215dbfc
  Cap maksimum run: $0.03
  Cap kumulatif sesi: $0.03


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:3LoMZljS4bDng3ilL] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:3LoMZljS4bDng3ilL] -> 2026-10-16T10:52:25.345Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:3LoMZljS4bDng3ilL] -> 2026-10-16T10:52:25.347Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:3LoMZljS4bDng3ilL] -> 2026-10-16T10:52:25.406Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:3LoMZljS4bDng3ilL] -> 2026-10-16T10:52:25.408Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:3LoMZljS4bDng3ilL] -> 2026-10-16T10:52:31.772Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  Raw: 22 | ID unik dalam run: 7 | Mock: 8
  Quality: MOCK_PRESENT
  Data quality: HAS_POLICY_CANDIDATES
  Kandidat policy: 7
  Consecutive no-real: 0
  Usage reported (USD): 0.0055

COLLECTION SESSION COMPLETE
Actor calls baru: 1
SAVED baru: 1
NEEDS_REVIEW baru: 0
Task ambigu lama dilewati: 0
Cap komitmen sesi: $0.03
Periksa saldo aktual Apify sebelum memulai sesi berikutnya.

One-run session finished.
Saved runs: 1
Review-needed runs: 0

RUNTIME PAID GATE LOCKED.
Pastikan Cell 04 juga dikembalikan ke konfigurasi OFF sebelum menyimpan.


In [61]:
# 11 — LAPORAN PROGRES GLOBAL (TANPA API): DE-DUP ID + TEMPORAL COVERAGE

def report_full_progress(mode='full'):
    state = load_checkpoint()
    planned = build_tasks(mode)
    planned_by_id = {t['task_id']: t for t in planned}
    statuses = Counter(state.get(tid, {}).get('status', 'PENDING') for tid in planned_by_id)
    print('\n=== FULL PROGRESS ===')
    print('Task direncanakan:', len(planned), '| status:', dict(statuses))
    by_case = {}
    all_ids = set()
    for case in FULL_CASES:
        case_ids = set()
        kept = 0
        mock = 0
        raw = 0
        missing = []
        for task in (t for t in planned if t['case'] == case):
            record = state.get(task['task_id'])
            if not record or record.get('status') != 'SAVED':
                continue
            rel = record.get('raw_path')
            if not rel:
                missing.append(task['task_id'])
                continue
            raw_path = PROJECT_ROOT / Path(str(rel).replace('\\', '/'))
            if not raw_path.is_file():
                missing.append(task['task_id'])
                continue
            with raw_path.open('r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip():
                        continue
                    obj = json.loads(line)
                    raw += 1
                    if obj.get('type') == 'mock_tweet':
                        mock += 1
                    if obj.get('type') == 'tweet' and obj.get('id') not in (None, -1, '-1'):
                        kept += 1
                        case_ids.add(str(obj['id']))
        all_ids.update(case_ids)
        by_case[case] = {
            'case': case, 'raw': raw, 'mock': mock, 'real_rows': kept,
            'unique_ids': len(case_ids), 'missing_saved_raw_files': len(missing),
        }
    display(pd.DataFrame(list(by_case.values())))
    print('ID asli unik global dalam task FULL yang sudah tersimpan:', len(all_ids))
    print('NOTE: sebagian run pilot non-6-jam tidak termasuk rencana FULL ini.')
    print('NOTE: ID unik belum tentu relevan semantik atau berlabel sentimen.')

    coverage = []
    for case in FULL_CASES:
        for day in sorted({t['start_wib'].date() for t in planned if t['case'] == case}):
            daily = [t for t in planned if t['case'] == case and t['start_wib'].date() == day]
            done = sum(state.get(t['task_id'], {}).get('status') == 'SAVED' for t in daily)
            coverage.append({
                'case': case, 'date_wib': str(day),
                'saved_windows': done, 'planned_windows': len(daily),
                'coverage_complete': done == len(daily),
            })
    df_coverage = pd.DataFrame(coverage)
    dest = REPORT_DIR / 'full_coverage_grid.csv'
    csv_write_atomic(dest, df_coverage)
    print('Coverage grid (offline) tersimpan:', dest)
    if not df_coverage.empty:
        print('Hari selesai 4/4 window:', int(df_coverage['coverage_complete'].sum()),
              'dari', len(df_coverage), 'hari-kasus')
    return by_case, df_coverage

# Cukup panggil report_full_progress() SETELAH cell tes berhasil,
# tanpa harus menjalankan scraping atau mengisi API token.


In [77]:
# 11A — LAPORAN KUALITAS GLOBAL: SEMUA RUN SAVED (PILOT + FULL)
# Dapat dijalankan tanpa API. Tidak mengubah status SAVED atau menghapus mock.

def report_global_data_quality():
    state = load_checkpoint()
    real_by_case = {case: set() for case in FULL_CASES}
    candidates_by_case = {case: set() for case in FULL_CASES}
    records = []
    for rec in state.values():
        if rec.get('status') != 'SAVED' or rec.get('case') not in FULL_CASES:
            continue
        case = rec['case']
        audit_rel = rec.get('audit_path')
        missing_audit = False
        if audit_rel:
            audit_path = PROJECT_ROOT / Path(str(audit_rel).replace('\\', '/'))
            if audit_path.is_file():
                df = pd.read_csv(audit_path, dtype={'id': 'string'}, keep_default_na=False)
                if 'id' in df and 'type' in df:
                    ids = df['id'].astype(str).str.strip()
                    real_mask = df['type'].eq('tweet') & ids.ne('') & ids.ne('-1')
                    real_by_case[case].update(ids.loc[real_mask].tolist())
                    if 'audit_candidate' in df:
                        candidate_mask = (
                            df['audit_candidate']
                            .astype(str)
                            .str.lower()
                            .eq('true')
                        )

                        # Koreksi historis khusus Palu:
                        # tweet bertopik Sulteng yang sebelumnya tidak dikenali
                        if case == 'palu_2018':
                            alias_match = df['text'].fillna('').astype(str).map(
                                lambda text: match_any_phrase(
                                    text,
                                    AUDIT_EVENT_ALIASES['palu_2018']
                                )
                            )

                            in_window = (
                                df['audit_in_window']
                                .astype(str)
                                .str.lower()
                                .eq('true')
                            )

                            policy_match = (
                                df['audit_has_policy']
                                .astype(str)
                                .str.lower()
                                .eq('true')
                            )

                            candidate_mask = (
                                candidate_mask
                                | (real_mask & in_window & policy_match & alias_match)
                            )

                        candidates_by_case[case].update(
                            ids.loc[real_mask & candidate_mask].tolist()
                        )
                else:
                    missing_audit = True
            else:
                missing_audit = True
        else:
            missing_audit = True
        quality = (
            'NO_REAL_DATA' if rec.get('unique_real_ids', 0) == 0 else
            'NO_POLICY_CANDIDATES' if rec.get('rows_event_and_policy_in_window', 0) == 0 else
            'HAS_POLICY_CANDIDATES'
        )
        records.append({
            'case': case, 'task_id': rec.get('task_id'), 'query_id': rec.get('query_id'),
            'status': rec['status'], 'data_quality_status': quality,
            'rows_raw': rec.get('rows_raw', 0), 'rows_mock': rec.get('rows_mock', 0),
            'unique_real_ids_in_run': rec.get('unique_real_ids', 0),
            'unique_policy_candidate_ids_in_run': rec.get('unique_policy_candidate_ids'),
            'audit_missing': missing_audit,
        })
    df_run = pd.DataFrame(records)
    summary = []
    for case in FULL_CASES:
        subset = df_run.loc[df_run['case'].eq(case)] if not df_run.empty else pd.DataFrame()
        summary.append({
            'case': case,
            'saved_tasks': len(subset),
            'saved_but_zero_real': int(subset['data_quality_status'].eq('NO_REAL_DATA').sum()) if len(subset) else 0,
            'global_unique_real_ids': len(real_by_case[case]),
            'global_unique_policy_candidates_auto': len(candidates_by_case[case]),
            'audit_files_missing': int(subset['audit_missing'].sum()) if len(subset) else 0,
        })
    df_summary = pd.DataFrame(summary)
    csv_write_atomic(REPORT_DIR / 'collection_quality_by_run.csv', df_run)
    csv_write_atomic(REPORT_DIR / 'collection_quality_by_case.csv', df_summary)
    display(df_summary)
    print('ID asli unik GLOBAL (semua kasus):', len(set().union(*real_by_case.values())))
    print('Kandidat keyword GLOBAL (perlu validasi semantik):',
        len(set().union(*candidates_by_case.values())))
    if df_summary['audit_files_missing'].sum() > 0:
        print('PERHATIAN: audit tidak lengkap. Jangan gunakan total unik ini sebagai angka final.')
    print('Catatan: SAVED = arsip sukses; kandidat otomatis bukan opini yang sudah diberi label.')
    return df_run, df_summary

# Jalankan manual setelah selesai batch: report_global_data_quality()


In [78]:
# 11B — LAPORAN KUALITAS GLOBAL
df_run_quality, df_case_quality = report_global_data_quality()

display(df_case_quality)

,case,saved_tasks,saved_but_zero_real,global_unique_real_ids,global_unique_policy_candidates_auto,audit_files_missing
0,lombok_2018,21,3,93,90,0
1,palu_2018,20,2,118,113,0
2,sumatra_2025,19,3,168,163,0


ID asli unik GLOBAL (semua kasus): 379
Kandidat keyword GLOBAL (perlu validasi semantik): 366
Catatan: SAVED = arsip sukses; kandidat otomatis bukan opini yang sudah diberi label.


,case,saved_tasks,saved_but_zero_real,global_unique_real_ids,global_unique_policy_candidates_auto,audit_files_missing
0,lombok_2018,21,3,93,90,0
1,palu_2018,20,2,118,113,0
2,sumatra_2025,19,3,168,163,0


In [79]:
# 11B.1 — BATCH 7 DYNAMIC MASTER + PROGRESS
# TARGET 1500 RAW / MINIMUM 400 UNIQUE IDs
# OFFLINE ONLY | SNAPSHOT 314 PRESERVED
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# A. SAFETY GUARD
# ------------------------------------------------------------

if EXECUTE_SCRAPING is not False:
    raise RuntimeError("Scraping harus OFF.")

if PAID_RUN_CONFIRMATION != "":
    raise RuntimeError("Paid confirmation harus kosong.")

if "batch7_scan_saved" not in globals():
    raise RuntimeError("Jalankan Cell 07D dahulu.")

if (TARGET_RAW, TARGET_UNIQUE_IDS) != (1500, 400):
    raise RuntimeError(
        "Target belum sesuai: wajib 1500 RAW / 400 ID."
    )

BATCH7_CURRENT_DIR = REPORT_DIR / "batch7" / "current"

FROZEN_FILES = [
    REPORT_DIR / "batch7" / "master_tweets_314.csv",
    REPORT_DIR / "batch7" / "raw_inventory_49_tasks.csv",
    REPORT_DIR / "batch7" / "baseline_manifest.json",
    REPORT_DIR / "batch7" / "manual_registry_314.csv",
]

for frozen_path in FROZEN_FILES:
    if not frozen_path.is_file():
        raise FileNotFoundError(frozen_path)

# ------------------------------------------------------------
# B. AUDIT CHECKPOINT DAN RAW
# ------------------------------------------------------------

scan = batch7_scan_saved()
state = scan["state"]

records = []
inventory = []

ordered_tasks = sorted(
    state.values(),
    key=lambda rec: (
        rec.get("saved_at_utc") or "",
        rec["task_id"]
    )
)

for rec in ordered_tasks:

    raw_path = PROJECT_ROOT / Path(
        str(rec["raw_path"]).replace("\\", "/")
    )

    if not raw_path.is_file():
        raise FileNotFoundError(raw_path)

    with raw_path.open(encoding="utf-8") as f:

        for line_no, line in enumerate(f, 1):

            if not line.strip():
                continue

            item = json.loads(line)

            author = item.get("author") or {}
            if not isinstance(author, dict):
                author = {}

            records.append({
                "case": rec["case"],
                "id": str(item.get("id", "")).strip(),
                "type": item.get("type", ""),
                "text": item.get("text") or "",
                "createdAt": item.get("createdAt") or "",
                "url": (
                    item.get("url")
                    or item.get("twitterUrl")
                    or ""
                ),
                "lang": item.get("lang") or "",
                "author_username": (
                    author.get("userName")
                    or author.get("username")
                    or ""
                ),
                "retweetCount": item.get("retweetCount"),
                "replyCount": item.get("replyCount"),
                "likeCount": item.get("likeCount"),
                "task_id": rec["task_id"],
                "run_id": rec.get("run_id", ""),
                "raw_path": str(rec["raw_path"]),
                "raw_line_number": line_no,
                "raw_payload_json": json.dumps(
                    item,
                    ensure_ascii=False,
                    default=str
                ),
            })

    inventory.append({
        "case": rec["case"],
        "task_id": rec["task_id"],
        "run_id": rec.get("run_id", ""),
        "raw_rows": rec["rows_raw"],
        "tweet_rows": rec["rows_real_tweet"],
        "mock_rows": rec["rows_mock"],
        "unique_ids_in_run": rec["unique_real_ids"],
        "raw_path": rec["raw_path"],
    })

# ------------------------------------------------------------
# C. MASTER DATASET — REAL TWEETS ONLY
# ------------------------------------------------------------

df_raw = pd.DataFrame(records)
df_inventory = pd.DataFrame(inventory)

df_real = df_raw.loc[
    df_raw["type"].eq("tweet")
].copy()

# ID Twitter dipertahankan sebagai string.
df_real["id"] = df_real["id"].astype(str)

df_master = (
    df_real
    .drop_duplicates("id", keep="first")
    .copy()
)

frequency = df_real.groupby("id").size()

source_tasks = (
    df_real.groupby("id")["task_id"]
    .agg(lambda values: "|".join(dict.fromkeys(values)))
)

df_master["occurrence_count"] = (
    df_master["id"].map(frequency)
)

df_master["source_task_ids"] = (
    df_master["id"].map(source_tasks)
)

df_master = (
    df_master
    .sort_values(
        ["case", "createdAt", "id"],
        kind="stable"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# D. KONSISTENSI DATA
# ------------------------------------------------------------

assert len(df_raw) == scan["raw_total"]
assert len(df_real) == scan["tweet_rows"]

assert (
    int(df_raw["type"].eq("mock_tweet").sum())
    == scan["mock_rows"]
)

assert len(df_master) == scan["unique_total"]
assert not df_master["id"].duplicated().any()
assert set(df_master["id"]) == scan["global_ids"]

assert (
    int(df_master["occurrence_count"].sum())
    == len(df_real)
)

assert len(df_inventory) == len(state)

# ------------------------------------------------------------
# E. PROGRESS PER KASUS
# ------------------------------------------------------------

summary_rows = []

for case in FULL_CASES:

    data = scan["by_case"][case]

    summary_rows.append({
        "case": case,
        "saved_tasks": data["tasks"],
        "raw_records": data["raw"],
        "real_tweet_rows": data["tweet_rows"],
        "mock_rows": data["mock_rows"],
        "unique_real_ids": len(data["ids"]),
    })

summary_rows.append({
    "case": "GLOBAL",
    "saved_tasks": len(state),
    "raw_records": scan["raw_total"],
    "real_tweet_rows": scan["tweet_rows"],
    "mock_rows": scan["mock_rows"],
    "unique_real_ids": scan["unique_total"],
})

df_progress = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# F. TARGET BARU
# ------------------------------------------------------------

raw_total = scan["raw_total"]
unique_total = scan["unique_total"]

remaining_raw = max(0, TARGET_RAW - raw_total)
remaining_unique = max(
    0, TARGET_UNIQUE_IDS - unique_total
)

raw_target_met = raw_total >= TARGET_RAW
unique_target_met = unique_total >= TARGET_UNIQUE_IDS

progress_total = pd.DataFrame([{
    "target_raw": TARGET_RAW,
    "target_unique_ids": TARGET_UNIQUE_IDS,
    "current_raw": raw_total,
    "current_unique_ids": unique_total,
    "remaining_raw": remaining_raw,
    "remaining_unique_ids": remaining_unique,
    "raw_target_met": raw_target_met,
    "unique_target_met": unique_target_met,
    "both_targets_met": (
        raw_target_met and unique_target_met
    ),
}])

# ------------------------------------------------------------
# G. VERIFIKASI CHECKPOINT TIDAK BERUBAH
# ------------------------------------------------------------

end_scan = batch7_scan_saved()

if (
    end_scan["checkpoint_sha"] != scan["checkpoint_sha"]
    or end_scan["raw_manifest_sha"]
       != scan["raw_manifest_sha"]
):
    raise RuntimeError(
        "RAW/checkpoint berubah selama audit. STOP."
    )

# ------------------------------------------------------------
# H. OUTPUT FILE DINAMIS
# ------------------------------------------------------------

BATCH7_CURRENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

master_output = (
    BATCH7_CURRENT_DIR / "master_tweets_current.csv"
)

inventory_output = (
    BATCH7_CURRENT_DIR / "raw_inventory_current.csv"
)

case_output = (
    BATCH7_CURRENT_DIR / "collection_progress_by_case.csv"
)

total_output = (
    BATCH7_CURRENT_DIR / "collection_progress_summary.csv"
)

manifest_output = (
    BATCH7_CURRENT_DIR / "current_manifest.json"
)

csv_write_atomic(master_output, df_master)
csv_write_atomic(inventory_output, df_inventory)
csv_write_atomic(case_output, df_progress)
csv_write_atomic(total_output, progress_total)

# ------------------------------------------------------------
# I. CURRENT MANIFEST — TARGET SINKRON
# ------------------------------------------------------------

manifest = {
    "report_version": "batch7_dynamic_v1",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "baseline_commit": (
        "cc808befabe47a79c02ff7064d603906bafdf6b5"
    ),
    "checkpoint_sha256": scan["checkpoint_sha"],
    "raw_manifest_sha256": scan["raw_manifest_sha"],
    "saved_tasks": len(state),
    "raw_total": raw_total,
    "tweet_rows": scan["tweet_rows"],
    "mock_rows": scan["mock_rows"],
    "unique_ids": unique_total,
    "target_raw": TARGET_RAW,
    "target_unique_ids": TARGET_UNIQUE_IDS,
}

with manifest_output.open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# J. VALIDASI FILE KELUARAN
# ------------------------------------------------------------

with manifest_output.open(encoding="utf-8") as f:
    saved_manifest = json.load(f)

assert saved_manifest["target_raw"] == 1500
assert saved_manifest["target_unique_ids"] == 400
assert saved_manifest["unique_ids"] == unique_total
assert saved_manifest["raw_total"] == raw_total

# ------------------------------------------------------------
# K. OUTPUT AUDIT
# ------------------------------------------------------------

print("=" * 65)
print("BATCH 7 DYNAMIC MASTER REPORT: PASS")
print("=" * 65)

print("RAW records       :", raw_total)
print("Tweet rows        :", scan["tweet_rows"])
print("Mock rows         :", scan["mock_rows"])
print("Unique tweet IDs  :", unique_total)
print("SAVED tasks       :", len(state))

print("TARGET RAW        :", TARGET_RAW)
print("TARGET UNIQUE     :", TARGET_UNIQUE_IDS)
print("REMAINING RAW     :", remaining_raw)
print("REMAINING UNIQUE  :", remaining_unique)

print("Stretch 500 IDs   :", max(0, 500 - unique_total))
print("Snapshot 314      : PRESERVED")
print("Apify calls       : 0")

display(df_progress)
display(progress_total)

print("Master:", master_output)
print("Manifest:", manifest_output)
print("=" * 65)


BATCH 7 DYNAMIC MASTER REPORT: PASS
RAW records       : 1401
Tweet rows        : 925
Mock rows         : 476
Unique tweet IDs  : 379
SAVED tasks       : 60
TARGET RAW        : 1500
TARGET UNIQUE     : 400
REMAINING RAW     : 99
REMAINING UNIQUE  : 21
Stretch 500 IDs   : 121
Snapshot 314      : PRESERVED
Apify calls       : 0


,case,saved_tasks,raw_records,real_tweet_rows,mock_rows,unique_real_ids
0,lombok_2018,21,420,216,204,93
1,palu_2018,20,450,276,174,118
2,sumatra_2025,19,531,433,98,168
3,GLOBAL,60,1401,925,476,379


,target_raw,target_unique_ids,current_raw,current_unique_ids,remaining_raw,remaining_unique_ids,raw_target_met,unique_target_met,both_targets_met
0,1500,400,1401,379,99,21,False,False,False


Master: C:\Penelitian Sentimen Bencana Nasional\01_Raw Data\outputs\reports\batch7\current\master_tweets_current.csv
Manifest: C:\Penelitian Sentimen Bencana Nasional\01_Raw Data\outputs\reports\batch7\current\current_manifest.json


In [80]:
# 11C — LAPORAN RELEVANSI PERISTIWA GLOBAL (OFFLINE)

def report_event_scope_v1():
    state = load_checkpoint()
    records = []
    missing_files = []

    for rec in state.values():
        if rec.get("status") != "SAVED":
            continue

        case_id = rec.get("case")
        if case_id not in DISASTER_CASES:
            continue

        audit_rel = rec.get("audit_path")
        audit_path = (
            PROJECT_ROOT / Path(str(audit_rel).replace("\\", "/"))
            if audit_rel else None
        )

        if audit_path is None or not audit_path.is_file():
            missing_files.append(str(audit_path))
            continue

        df = pd.read_csv(
            audit_path,
            dtype={"id": "string"},
            keep_default_na=False,
        )

        required = {"id", "type", "text", "audit_in_window"}
        missing_cols = required - set(df.columns)

        if missing_cols:
            raise ValueError(
                f"Kolom audit hilang pada {audit_path}: {missing_cols}"
            )

        ids = df["id"].astype(str).str.strip()
        in_window = (
            df["audit_in_window"]
            .astype(str)
            .str.lower()
            .eq("true")
        )

        valid = (
            df["type"].eq("tweet")
            & ids.ne("")
            & ids.ne("-1")
            & in_window
        )

        event_terms = get_audit_event_terms(case_id)

        # Alias Sumatra hanya untuk audit turunan,
        # bukan untuk mengubah query scraping.
        if case_id == "sumatra_2025":
            event_terms = (
                event_terms + SUMATRA_SCOPE_LOCATIONS
            )

        policy_terms = DISASTER_CASES[case_id]["policy_terms"]

        for item in df.loc[valid].to_dict("records"):
            text = str(item["text"])

            status, reason = classify_event_scope(
                text, case_id
            )

            keyword_candidate = (
                match_any_phrase(text, event_terms)
                and match_any_phrase(text, policy_terms)
            )

            records.append({
                "audit_version": EVENT_SCOPE_AUDIT_VERSION,
                "case": case_id,
                "id": str(item["id"]).strip(),
                "text": text,
                "keyword_candidate": keyword_candidate,
                "event_scope": status,
                "scope_reason": reason,
                "task_id": rec["task_id"],
                "run_id": rec.get("run_id"),
            })

    # Jangan menulis laporan parsial jika file audit hilang.
    if missing_files:
        raise RuntimeError(
            f"{len(missing_files)} audit CSV tidak ditemukan. "
            f"Contoh: {missing_files[:3]}"
        )

    if not records:
        raise RuntimeError(
            "Belum ada tweet asli dalam audit CSV."
        )

    detail = pd.DataFrame(records)

    # Utamakan rekaman yang memenuhi keyword dan
    # status yang lebih informatif saat ada ID duplikat.
    rank = {
        "TARGET_WITH_HISTORY": 4,
        "TARGET_CURRENT": 3,
        "NEEDS_REVIEW": 2,
        "HISTORICAL_ONLY": 1,
    }

    detail["_rank"] = (
        detail["keyword_candidate"].astype(int) * 10
        + detail["event_scope"].map(rank).fillna(0)
    )

    detail = (
        detail.sort_values(
            "_rank", ascending=False, kind="stable"
        )
        .drop_duplicates(["case", "id"])
        .drop(columns="_rank")
        .reset_index(drop=True)
    )

    summary = []

    for case_id in DISASTER_CASES:
        case_df = detail.loc[detail["case"].eq(case_id)]
        candidates = case_df.loc[
            case_df["keyword_candidate"]
        ]

        counts = candidates["event_scope"].value_counts()

        current = int(counts.get("TARGET_CURRENT", 0))
        with_history = int(
            counts.get("TARGET_WITH_HISTORY", 0)
        )

        summary.append({
            "case": case_id,
            "unique_real_in_window": len(case_df),
            "keyword_candidates_rechecked": len(candidates),
            "target_current": current,
            "target_with_history": with_history,
            "historical_only": int(
                counts.get("HISTORICAL_ONLY", 0)
            ),
            "needs_review": int(
                counts.get("NEEDS_REVIEW", 0)
            ),
            "event_relevant_auto": current + with_history,
        })

    df_summary = pd.DataFrame(summary)

    csv_write_atomic(
        REPORT_DIR / "event_scope_v1_by_tweet.csv",
        detail
    )

    csv_write_atomic(
        REPORT_DIR / "event_scope_v1_by_case.csv",
        df_summary
    )

    display(df_summary)

    print(
        "Tweet unik (dalam window):",
        len(detail)
    )
    print(
        "Kandidat peristiwa otomatis:",
        df_summary["event_relevant_auto"].sum()
    )
    print(
        "Perlu pemeriksaan manual:",
        df_summary["needs_review"].sum()
    )

    return detail, df_summary


df_event_scope, df_event_scope_summary = report_event_scope_v1()


,case,unique_real_in_window,keyword_candidates_rechecked,target_current,target_with_history,historical_only,needs_review,event_relevant_auto
0,lombok_2018,93,90,90,0,0,0,90
1,palu_2018,118,113,112,1,0,0,113
2,sumatra_2025,168,163,100,1,28,34,101


Tweet unik (dalam window): 379
Kandidat peristiwa otomatis: 304
Perlu pemeriksaan manual: 34


In [81]:
# 11D — EVENT SCOPE V2: LAPORAN, PERBANDINGAN V1, DAN REVIEW MANUAL
# Letakkan setelah Cell 11C. Jalankan saat EXECUTE_SCRAPING == False.
# Input: event_scope_v1_by_tweet.csv (TIDAK ditimpa).
from IPython.display import display
import hashlib


def report_event_scope_v2():
    v1_file = REPORT_DIR / "event_scope_v1_by_tweet.csv"
    if not v1_file.is_file():
        raise FileNotFoundError(
            "Laporan V1 tidak ditemukan. Jalankan Cell 11C (OFFLINE) lebih dahulu."
        )

    df = pd.read_csv(
        v1_file, dtype={"id": "string"}, keep_default_na=False
    )
    needed = {
        "case", "id", "text", "keyword_candidate", "event_scope",
        "scope_reason", "task_id", "run_id",
    }
    if missing := needed - set(df.columns):
        raise ValueError(f"Kolom wajib V1 hilang: {sorted(missing)}")
    if df.duplicated(["case", "id"]).any():
        raise ValueError("File V1 memiliki duplikasi (case, id). Periksa sebelum lanjut.")

    # Simpan hasil V1 persis sebagai basis perbandingan; jangan tulis ulang CSV V1.
    df = df.rename(columns={
        "event_scope": "v1_scope", "scope_reason": "v1_reason"
    }).copy()
    df["keyword_candidate"] = (
        df["keyword_candidate"].astype(str).str.lower().eq("true")
    )
    df["v2_version"] = EVENT_SCOPE_V2_VERSION

    v2 = [
        classify_event_scope_v2(row.text, row.case)
        for row in df.itertuples(index=False)
    ]
    df["v2_scope"] = [x[0] for x in v2]
    df["v2_reason"] = [x[1] for x in v2]
    df["v1_to_v2_changed"] = df["v1_scope"].ne(df["v2_scope"])

    # Tambah waktu unggahan dari audit asli berdasarkan (case, id).
    # V1 tetap apa adanya; raw/checkpoint/audit asli tidak ditimpa.
    state = load_checkpoint()
    created_at = {}
    missing_audits = []
    for task in state.values():
        if task.get("status") != "SAVED":
            continue
        path = task.get("audit_path")
        if not path:
            missing_audits.append(str(task.get("task_id")))
            continue
        audit_file = PROJECT_ROOT / Path(str(path).replace("\\", "/"))
        if not audit_file.is_file():
            missing_audits.append(str(audit_file))
            continue
        source = pd.read_csv(
            audit_file,
            dtype={"id": "string"},
            keep_default_na=False,
            usecols=lambda c: c in ("id", "type", "createdAt"),
        )
        if not {"id", "type", "createdAt"}.issubset(source.columns):
            raise ValueError(f"Kolom id/type/createdAt hilang: {audit_file}")
        for row in source.loc[source["type"].eq("tweet")].itertuples(index=False):
            key = (str(task["case"]), str(row.id).strip())
            if key not in created_at and str(row.createdAt).strip():
                created_at[key] = str(row.createdAt)

    if missing_audits:
        raise RuntimeError(
            f"{len(missing_audits)} audit historis hilang; "
            f"contoh: {missing_audits[:2]}"
        )

    df["created_at_source"] = [
        created_at.get((str(row.case), str(row.id)), "")
        for row in df.itertuples(index=False)
    ]

    # Ringkasan hanya untuk tweet yang lolos keyword V1 (agar sebanding).
    cand = df.loc[df["keyword_candidate"]].copy()
    cases = list(DISASTER_CASES)
    summary = []
    for case in cases:
        group = cand.loc[cand["case"].eq(case)]
        all_case = df.loc[df["case"].eq(case)]
        counts = group["v2_scope"].value_counts()
        summary.append({
            "case": case,
            "unique_real": len(all_case),
            "keyword_candidates": len(group),
            "target_current": int(counts.get("TARGET_CURRENT", 0)),
            "target_with_history": int(counts.get("TARGET_WITH_HISTORY", 0)),
            "historical_only": int(counts.get("HISTORICAL_ONLY", 0)),
            "needs_review": int(counts.get("NEEDS_REVIEW", 0)),
            "event_relevant_auto": int(
                counts.get("TARGET_CURRENT", 0)
                + counts.get("TARGET_WITH_HISTORY", 0)
            ),
            "changed_vs_v1": int(group["v1_to_v2_changed"].sum()),
        })
    by_case = pd.DataFrame(summary)

    transitions = (
        cand.groupby(["case", "v1_scope", "v2_scope"], dropna=False)
        .size().reset_index(name="n_tweets")
        .sort_values(["case", "v1_scope", "v2_scope"])
    )

    assert len(df) == df[["case", "id"]].drop_duplicates().shape[0]
    assert int(by_case["unique_real"].sum()) == len(df)
    assert int(by_case["keyword_candidates"].sum()) == len(cand)
    assert (by_case[[
        "target_current", "target_with_history", "historical_only", "needs_review"
    ]].sum(axis=1) == by_case["keyword_candidates"]).all()

    # Simpan V2 dengan nama baru; JANGAN overwrite V1.
    csv_write_atomic(REPORT_DIR / "event_scope_v2_by_tweet.csv", df)
    csv_write_atomic(REPORT_DIR / "event_scope_v2_by_case.csv", by_case)
    csv_write_atomic(REPORT_DIR / "event_scope_v2_transitions.csv", transitions)

    # Review prioritas: semua ambigu/historis/perubahan dari V1.
    # Tambah kontrol deterministik ~20% dari label yang tidak berubah,
    # untuk memeriksa false positive yang lolos di kedua versi.
    queue_path = REPORT_DIR / "event_scope_v2_manual_review.csv"
    priority_mask = (
        cand["v2_scope"].isin(["NEEDS_REVIEW", "HISTORICAL_ONLY"])
        | cand["v1_to_v2_changed"]
    )
    unchanged_target = cand["v2_scope"].isin([
        "TARGET_CURRENT", "TARGET_WITH_HISTORY"
    ]) & ~cand["v1_to_v2_changed"]
    sample_unchanged = pd.Series([
        int(hashlib.sha256(
            f"{r.case}:{r.id}".encode("utf-8")
        ).hexdigest()[:8], 16) % 5 == 0
        for r in cand.itertuples(index=False)
    ], index=cand.index)
    queue_mask = priority_mask | (unchanged_target & sample_unchanged)
    queue = cand.loc[queue_mask, [
        "case", "id", "created_at_source", "text", "v1_scope", "v2_scope",
        "v2_reason", "v1_to_v2_changed",
    ]].copy()
    queue["review_stratum"] = [
        "priority" if priority_mask.loc[i] else "sample_unchanged"
        for i in queue.index
    ]
    queue["manual_scope"] = ""
    queue["manual_notes"] = ""
    queue["reviewer"] = ""
    queue["reviewed_at"] = ""
    if queue_path.exists():
        prev = pd.read_csv(queue_path, dtype={"id": "string"}, keep_default_na=False)
        if prev.duplicated(["case", "id"]).any():
            raise ValueError("Ada ID ganda dalam antrean review lama.")
        old_ids = set(zip(prev["case"].astype(str), prev["id"].astype(str)))
        fresh = queue.loc[[
            (str(r.case), str(r.id)) not in old_ids
            for r in queue.itertuples(index=False)
        ]]
        # Baris lama tidak diubah agar label dan catatan manual aman.
        queue = pd.concat([prev, fresh], ignore_index=True)
    csv_write_atomic(queue_path, queue)

    display(by_case)
    display(transitions)
    print("ID unik asli:", len(df))
    print("Kandidat keyword:", len(cand))
    print("Relevan otomatis V2:", int(by_case["event_relevant_auto"].sum()))
    print("Berubah dari V1:", int(cand["v1_to_v2_changed"].sum()))
    print("Antrean review (termasuk label yang sudah dikerjakan):", len(queue))
    return df, by_case, transitions, queue


df_scope_v2, df_scope_v2_case, df_scope_v2_transitions, df_scope_v2_review = report_event_scope_v2()


,case,unique_real,keyword_candidates,target_current,target_with_history,historical_only,needs_review,event_relevant_auto,changed_vs_v1
0,lombok_2018,93,90,90,0,0,0,90,0
1,palu_2018,118,113,112,1,0,0,113,0
2,sumatra_2025,168,163,127,4,13,19,131,43


,case,v1_scope,v2_scope,n_tweets
0,lombok_2018,TARGET_CURRENT,TARGET_CURRENT,90
1,palu_2018,TARGET_CURRENT,TARGET_CURRENT,112
2,palu_2018,TARGET_WITH_HISTORY,TARGET_WITH_HISTORY,1
3,sumatra_2025,HISTORICAL_ONLY,HISTORICAL_ONLY,13
4,sumatra_2025,HISTORICAL_ONLY,NEEDS_REVIEW,13
5,sumatra_2025,HISTORICAL_ONLY,TARGET_WITH_HISTORY,2
6,sumatra_2025,NEEDS_REVIEW,NEEDS_REVIEW,6
7,sumatra_2025,NEEDS_REVIEW,TARGET_CURRENT,27
8,sumatra_2025,NEEDS_REVIEW,TARGET_WITH_HISTORY,1
9,sumatra_2025,TARGET_CURRENT,TARGET_CURRENT,100


ID unik asli: 379
Kandidat keyword: 366
Relevan otomatis V2: 334
Berubah dari V1: 43
Antrean review (termasuk label yang sudah dikerjakan): 132


In [27]:
import sys
import openpyxl

print("Python:", sys.executable)
print("Openpyxl:", openpyxl.__version__)

Python: c:\Penelitian Sentimen Bencana Nasional\.venv\Scripts\python.exe
Openpyxl: 3.1.5


In [82]:
# 11E — BATCH 7 DYNAMIC MANUAL REGISTRY
# Preserve historical manual labels, no Apify
# ============================================================

import pandas as pd

assert EXECUTE_SCRAPING is False
assert PAID_RUN_CONFIRMATION == ""

V2_FILE = REPORT_DIR / "event_scope_v2_by_tweet.csv"

MASTER_FILE = (
    REPORT_DIR / "batch7" / "current"
    / "master_tweets_current.csv"
)

FROZEN_MANUAL = (
    REPORT_DIR / "batch7"
    / "manual_registry_314.csv"
)

CURRENT_MANUAL = (
    REPORT_DIR / "batch7" / "current"
    / "manual_registry_current.csv"
)

for path in [V2_FILE, MASTER_FILE, FROZEN_MANUAL]:
    if not path.is_file():
        raise FileNotFoundError(path)

def read_ids(path):
    df = pd.read_csv(
        path,
        dtype={"id": "string"},
        keep_default_na=False
    )
    df["case"] = df["case"].astype(str).str.strip()
    df["id"] = df["id"].astype(str).str.strip()

    if df.duplicated(["case", "id"]).any():
        raise RuntimeError(
            "Duplikasi ID pada: " + str(path)
        )
    return df

v2 = read_ids(V2_FILE)
master = read_ids(MASTER_FILE)

key_cols = ["case", "id"]

v2_keys = set(zip(v2.case, v2.id))
master_keys = set(zip(master.case, master.id))

if v2_keys != master_keys:
    raise RuntimeError(
        "Event Scope V2 belum sesuai master terbaru. "
        "Jalankan ulang Cell 11C dan 11D."
    )

# Sumber keputusan manual:
# 1. Snapshot 314 historis
# 2. Registry current sebelumnya (jika ada)
sources = [read_ids(FROZEN_MANUAL)]

if CURRENT_MANUAL.is_file():
    sources.append(read_ids(CURRENT_MANUAL))

manual_by_key = {}

for source in sources:

    needed = {
        "case", "id", "text",
        "manual_inclusion"
    }

    if needed - set(source.columns):
        raise RuntimeError(
            "Kolom manual tidak lengkap."
        )

    for row in source.to_dict("records"):

        key = (
            str(row["case"]),
            str(row["id"])
        )

        decision = str(
            row.get("manual_inclusion", "")
        ).strip().upper()

        if decision not in ("", "INCLUDE", "EXCLUDE"):
            raise RuntimeError(
                "Keputusan manual tidak dikenal."
            )

        if not decision:
            continue

        if key not in v2_keys:
            raise RuntimeError(
                "ID manual hilang dari master terbaru: "
                + str(key)
            )

        prior = manual_by_key.get(key)

        if (
            prior is not None
            and prior["manual_inclusion"] != decision
        ):
            raise RuntimeError(
                "Konflik perubahan keputusan manual: "
                + str(key)
            )

        manual_by_key[key] = {
            "manual_inclusion": decision,
            "manual_scope": row.get(
                "manual_scope", ""
            ),
            "manual_notes": row.get(
                "manual_notes", ""
            ),
            "manual_event_scope": row.get(
                "manual_event_scope", ""
            ),
            "reviewer": row.get(
                "reviewer", ""
            ),
            "reviewed_at": row.get(
                "reviewed_at", ""
            ),
            "text_original": str(
                row.get("text", "")
            ),
        }

# Minimal 84 keputusan historis wajib tersedia.
if len(manual_by_key) < 84:
    raise RuntimeError(
        "Keputusan manual historis berkurang."
    )

manual_rows = []

for row in v2.to_dict("records"):

    key = (
        str(row["case"]),
        str(row["id"])
    )

    decision = manual_by_key.get(key)

    if decision:

        if str(row["text"]) != decision["text_original"]:
            raise RuntimeError(
                "Teks tweet berlabel berubah: "
                + str(key)
            )

        manual_rows.append({
            "case": key[0],
            "id": key[1],
            **{
                field: decision[field]
                for field in (
                    "manual_inclusion",
                    "manual_scope",
                    "manual_notes",
                    "manual_event_scope",
                    "reviewer",
                    "reviewed_at"
                )
            }
        })

manual_df = pd.DataFrame(
    manual_rows,
    columns=[
        "case", "id", "manual_inclusion",
        "manual_scope", "manual_notes",
        "manual_event_scope", "reviewer",
        "reviewed_at"
    ]
)

registry = v2.merge(
    manual_df,
    on=["case", "id"],
    how="left",
    validate="one_to_one"
)

for col in manual_df.columns:
    if col not in ("case", "id"):
        registry[col] = registry[col].fillna("")

registry["final_inclusion_decision"] = (
    registry["manual_inclusion"].where(
        registry["manual_inclusion"].isin(
            ["INCLUDE", "EXCLUDE"]
        ),
        "PENDING_MANUAL"
    )
)

registry["manual_review_status"] = (
    registry["final_inclusion_decision"].map({
        "INCLUDE": "REVIEWED",
        "EXCLUDE": "REVIEWED",
        "PENDING_MANUAL": "NOT_REVIEWED"
    })
)

assert len(registry) == len(master)

csv_write_atomic(CURRENT_MANUAL, registry)

print("BATCH 7 MANUAL REGISTRY: PASS")
print("Total IDs:", len(registry))
print(
    "Reviewed:",
    int(registry["manual_review_status"]
        .eq("REVIEWED").sum())
)
print(
    "Pending:",
    int(registry["manual_review_status"]
        .eq("NOT_REVIEWED").sum())
)

display(pd.crosstab(
    registry["case"],
    registry["final_inclusion_decision"]
))


BATCH 7 MANUAL REGISTRY: PASS
Total IDs: 379
Reviewed: 84
Pending: 295


final_inclusion_decision,EXCLUDE,INCLUDE,PENDING_MANUAL
case,,,
lombok_2018,2,10,81
palu_2018,1,8,109
sumatra_2025,14,49,105


In [83]:
# 11F — BATCH 7 DYNAMIC POLICY FOCUS V2.1
# OFFLINE ONLY
# ============================================================

import pandas as pd

assert EXECUTE_SCRAPING is False
assert PAID_RUN_CONFIRMATION == ""

V2_FILE = (
    REPORT_DIR / "event_scope_v2_by_tweet.csv"
)

REGISTRY_FILE = (
    REPORT_DIR / "batch7" / "current"
    / "manual_registry_current.csv"
)

MASTER_FILE = (
    REPORT_DIR / "batch7" / "current"
    / "master_tweets_current.csv"
)

for path in [V2_FILE, REGISTRY_FILE, MASTER_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

def load_policy_data(path):
    df = pd.read_csv(
        path,
        dtype={"id": "string"},
        keep_default_na=False
    )
    df["case"] = df["case"].astype(str).str.strip()
    df["id"] = df["id"].astype(str).str.strip()

    if df.duplicated(["case", "id"]).any():
        raise RuntimeError(
            "Duplikasi ID pada " + str(path)
        )

    return df

v2 = load_policy_data(V2_FILE)
registry = load_policy_data(REGISTRY_FILE)
master = load_policy_data(MASTER_FILE)

keys = lambda d: set(
    zip(d["case"], d["id"])
)

if not (
    keys(v2) == keys(registry) == keys(master)
):
    raise RuntimeError(
        "ID di master, V2, dan manual registry berbeda."
    )

manual_cols = [
    "case", "id", "manual_inclusion",
    "manual_scope", "manual_notes",
    "manual_event_scope", "reviewer",
    "reviewed_at"
]

result = v2.merge(
    registry[manual_cols],
    on=["case", "id"],
    how="left",
    validate="one_to_one"
)

for col in manual_cols:
    if col not in ("case", "id"):
        result[col] = result[col].fillna("")

result["manual_inclusion"] = (
    result["manual_inclusion"]
    .astype(str)
    .str.strip()
    .str.upper()
)

valid = {"", "INCLUDE", "EXCLUDE"}

if not result["manual_inclusion"].isin(valid).all():
    raise RuntimeError(
        "Keputusan manual tidak valid."
    )


# ------------------------------------------------------------
# KLASIFIKASI FOKUS KEBIJAKAN
# ------------------------------------------------------------

predictions = [
    classify_policy_focus_v21(
        row.text,
        row.case,
        row.v2_scope
    )
    for row in result.itertuples(index=False)
]

result["policy_focus_version"] = (
    POLICY_FOCUS_VERSION
)

result["policy_focus_v21"] = [
    x[0] for x in predictions
]

result["policy_focus_reason_v21"] = [
    x[1] for x in predictions
]

result["final_inclusion_decision"] = (
    result["manual_inclusion"].where(
        result["manual_inclusion"].isin(
            ["INCLUDE", "EXCLUDE"]
        ),
        "PENDING_MANUAL"
    )
)

suggestions = {
    "INCLUDE_CANDIDATE": "INCLUDE",
    "EXCLUDE_CANDIDATE": "EXCLUDE"
}

result["v21_binary_suggestion"] = (
    result["policy_focus_v21"]
    .map(suggestions)
    .fillna("ABSTAIN")
)

result["manual_vs_v21_conflict"] = (
    result["final_inclusion_decision"].isin(
        ["INCLUDE", "EXCLUDE"]
    )
    & result["v21_binary_suggestion"].isin(
        ["INCLUDE", "EXCLUDE"]
    )
    & result["final_inclusion_decision"].ne(
        result["v21_binary_suggestion"]
    )
)


# ------------------------------------------------------------
# LAPORAN PER KASUS
# ------------------------------------------------------------

rows = []

for case, group in result.groupby("case"):

    focus = group[
        "policy_focus_v21"
    ].value_counts()

    manual = group[
        "final_inclusion_decision"
    ].value_counts()

    rows.append({
        "case": case,
        "unique_real_ids": len(group),
        "keyword_candidates": int(
            group["keyword_candidate"]
            .astype(str)
            .str.lower()
            .eq("true")
            .sum()
        ),
        "manual_include": int(
            manual.get("INCLUDE", 0)
        ),
        "manual_exclude": int(
            manual.get("EXCLUDE", 0)
        ),
        "pending_manual": int(
            manual.get("PENDING_MANUAL", 0)
        ),
        "v21_include_candidate": int(
            focus.get("INCLUDE_CANDIDATE", 0)
        ),
        "v21_exclude_candidate": int(
            focus.get("EXCLUDE_CANDIDATE", 0)
        ),
        "v21_needs_context": int(
            focus.get("NEEDS_CONTEXT", 0)
        ),
        "manual_vs_v21_conflict": int(
            group["manual_vs_v21_conflict"].sum()
        ),
    })

summary = pd.DataFrame(rows)

conflicts = result.loc[
    result["manual_vs_v21_conflict"],
    [
        "case", "id", "text",
        "v2_scope", "policy_focus_v21",
        "policy_focus_reason_v21",
        "manual_inclusion", "reviewed_at"
    ]
].copy()

assert len(result) == len(master)

assert int(
    summary["unique_real_ids"].sum()
) == len(master)

assert int(
    summary[
        [
            "manual_include",
            "manual_exclude",
            "pending_manual"
        ]
    ].sum().sum()
) == len(master)


# ------------------------------------------------------------
# OUTPUT DINAMIS
# ------------------------------------------------------------

csv_write_atomic(
    REPORT_DIR / "policy_focus_v21_by_tweet.csv",
    result
)

csv_write_atomic(
    REPORT_DIR / "policy_focus_v21_by_case.csv",
    summary
)

csv_write_atomic(
    REPORT_DIR / "policy_focus_v21_manual_conflicts.csv",
    conflicts
)

print("=" * 65)
print("BATCH 7 POLICY FOCUS V2.1: PASS")
print("=" * 65)

print("Unique IDs:", len(result))

print(
    "Manual reviewed:",
    int(result["final_inclusion_decision"]
        .isin(["INCLUDE", "EXCLUDE"]).sum())
)

print(
    "Pending manual:",
    int(result["final_inclusion_decision"]
        .eq("PENDING_MANUAL").sum())
)

print("Conflicts:", len(conflicts))

display(summary)


BATCH 7 POLICY FOCUS V2.1: PASS
Unique IDs: 379
Manual reviewed: 84
Pending manual: 295
Conflicts: 0


,case,unique_real_ids,keyword_candidates,manual_include,manual_exclude,pending_manual,v21_include_candidate,v21_exclude_candidate,v21_needs_context,manual_vs_v21_conflict
0,lombok_2018,93,90,10,2,81,60,4,29,0
1,palu_2018,118,113,8,1,109,86,3,29,0
2,sumatra_2025,168,163,49,14,105,89,8,71,0


In [67]:
# 11G — ONE-RUN YIELD CHECK (OFFLINE)
import json
from pathlib import Path

assert EXECUTE_SCRAPING is False

target_id = "sumatra_2025_70a5cb5da2f03ddd0761"
state = load_checkpoint()

old_ids = set()
test_ids = set()

for rec in state.values():
    if rec.get("status") != "SAVED":
        continue

    path = PROJECT_ROOT / Path(
        rec["raw_path"].replace("\\", "/")
    )

    assert path.is_file(), f"RAW hilang: {path}"

    id_set = (
        test_ids
        if rec["task_id"] == target_id
        else old_ids
    )

    with path.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)

            if item.get("type") != "tweet":
                continue

            tid = str(item.get("id", "")).strip()

            if tid not in ("", "-1", "None"):
                id_set.add(tid)

assert len(old_ids) == 232, (
    f"Baseline berubah: {len(old_ids)}"
)

new_ids = test_ids - old_ids
total_ids = old_ids | test_ids

print("BASELINE ID UNIK:", len(old_ids))
print("ID UNIK DALAM ONE-RUN:", len(test_ids))
print("ID BARU NON-DUPLIKAT:", len(new_ids))
print("TOTAL ID UNIK:", len(total_ids))
print("SISA MENUJU 300:", max(0, 300-len(total_ids)))
print("SISA MENUJU 350:", max(0, 350-len(total_ids)))
print("TOTAL TASK CHECKPOINT:", len(state))

if target_id not in state:
    print("STATUS: Task belum dieksekusi.")
else:
    print("STATUS:", state[target_id]["status"])


BASELINE ID UNIK: 232
ID UNIK DALAM ONE-RUN: 12
ID BARU NON-DUPLIKAT: 12
TOTAL ID UNIK: 244
SISA MENUJU 300: 56
SISA MENUJU 350: 106
TOTAL TASK CHECKPOINT: 43
STATUS: SAVED


In [84]:
# 11H — BATCH 7 DYNAMIC YIELD AUDIT
# TARGET: 1500 RAW + 400 UNIQUE IDs
# STRETCH: 500 UNIQUE IDs
# OFFLINE ONLY
# ============================================================

assert EXECUTE_SCRAPING is False
assert PAID_RUN_CONFIRMATION == ""

if (TARGET_RAW, TARGET_UNIQUE_IDS) != (1500, 400):
    raise RuntimeError("Target tidak sesuai.")

scan = batch7_scan_saved()
state = scan["state"]

if not state:
    raise RuntimeError("Checkpoint kosong.")

# ------------------------------------------------------------
# A. IDENTIFIKASI RUN TERAKHIR
# ------------------------------------------------------------

last = max(
    state.values(),
    key=lambda r: r.get("saved_at_utc") or ""
)

task_id = last["task_id"]

last_ids = scan["ids_by_task"][task_id]

previous_ids = set()

for previous_task_id, ids in (
    scan["ids_by_task"].items()
):
    if previous_task_id != task_id:
        previous_ids.update(ids)

new_ids = last_ids - previous_ids

# ------------------------------------------------------------
# B. PERHITUNGAN TARGET
# ------------------------------------------------------------

raw_total = scan["raw_total"]
unique_total = scan["unique_total"]

target_raw = TARGET_RAW
target_unique = TARGET_UNIQUE_IDS
stretch_unique = 500

remaining_raw = max(
    0, target_raw - raw_total
)

remaining_unique = max(
    0, target_unique - unique_total
)

remaining_stretch = max(
    0, stretch_unique - unique_total
)

raw_met = raw_total >= target_raw
unique_met = unique_total >= target_unique
both_met = raw_met and unique_met

# ------------------------------------------------------------
# C. OUTPUT RUN TERAKHIR
# ------------------------------------------------------------

print("=" * 65)
print("BATCH 7 DYNAMIC YIELD AUDIT")
print("=" * 65)

print("LAST TASK        :", task_id)
print("CASE             :", last["case"])
print("ACTOR STATUS     :", last["actor_status"])

print("RAW LAST RUN     :", last["rows_raw"])
print("REAL LAST RUN    :", last["rows_real_tweet"])
print("MOCK LAST RUN    :", last["rows_mock"])

print("NEW UNIQUE IDs   :", len(new_ids))

# ------------------------------------------------------------
# D. PROGRESS GLOBAL
# ------------------------------------------------------------

print("\nPROGRESS GLOBAL")
print("-" * 65)

print("TOTAL RAW        :", raw_total)
print("TOTAL REAL ROWS  :", scan["tweet_rows"])
print("TOTAL MOCK       :", scan["mock_rows"])
print("TOTAL UNIQUE IDs :", unique_total)
print("SAVED TASKS      :", len(state))

print("\nTARGET")
print("-" * 65)

print("TARGET RAW       :", target_raw)
print("TARGET UNIQUE    :", target_unique)
print("STRETCH UNIQUE   :", stretch_unique)

print("REMAINING RAW    :", remaining_raw)
print("REMAINING UNIQUE :", remaining_unique)
print("REMAINING STRETCH:", remaining_stretch)

# ------------------------------------------------------------
# E. DISTRIBUSI PER KASUS
# ------------------------------------------------------------

print("\nDISTRIBUSI PER KASUS")
print("-" * 65)

for case, data in scan["by_case"].items():
    print(
        case,
        ":",
        len(data["ids"]),
        "ID unik"
    )

# ------------------------------------------------------------
# F. KEPUTUSAN
# ------------------------------------------------------------

print("\nSTATUS")
print("-" * 65)

if both_met:
    print(
        "MINIMUM TARGETS ACHIEVED — "
        "STOP PAID SCRAPING AND REVIEW DATA."
    )

elif len(new_ids) == 0:
    print(
        "ZERO NEW UNIQUE IDs — "
        "PAUSE AND REVIEW YIELD."
    )

else:
    print(
        "TARGET NOT YET COMPLETE — "
        "REVIEW BEFORE NEXT RUN."
    )

print("=" * 65)


BATCH 7 DYNAMIC YIELD AUDIT
LAST TASK        : lombok_2018_ead812dc9b218215dbfc
CASE             : lombok_2018
ACTOR STATUS     : SUCCEEDED
RAW LAST RUN     : 22
REAL LAST RUN    : 14
MOCK LAST RUN    : 8
NEW UNIQUE IDs   : 7

PROGRESS GLOBAL
-----------------------------------------------------------------
TOTAL RAW        : 1401
TOTAL REAL ROWS  : 925
TOTAL MOCK       : 476
TOTAL UNIQUE IDs : 379
SAVED TASKS      : 60

TARGET
-----------------------------------------------------------------
TARGET RAW       : 1500
TARGET UNIQUE    : 400
STRETCH UNIQUE   : 500
REMAINING RAW    : 99
REMAINING UNIQUE : 21
REMAINING STRETCH: 121

DISTRIBUSI PER KASUS
-----------------------------------------------------------------
lombok_2018 : 93 ID unik
palu_2018 : 118 ID unik
sumatra_2025 : 168 ID unik

STATUS
-----------------------------------------------------------------
TARGET NOT YET COMPLETE — REVIEW BEFORE NEXT RUN.


In [72]:
# FINAL VALIDATION — PREFLIGHT PART 12
# OFFLINE ONLY

assert EXECUTE_SCRAPING is False
assert PAID_RUN_CONFIRMATION == ""

scan = batch7_scan_saved()

assert len(scan["state"]) == 59
assert scan["raw_total"] == 1379
assert scan["unique_total"] == 372

assert len(recent) == 10
assert all(
    rec["status"] == "SAVED"
    for rec in scan["state"].values()
)

print("=" * 60)
print("BATCH 7 — PART 12 OFFLINE VALIDATION")
print("=" * 60)
print("Status       : PASS")
print("Saved tasks  :", len(scan["state"]))
print("RAW total    :", scan["raw_total"])
print("Unique IDs   :", scan["unique_total"])
print("Remaining RAW:", 1500 - scan["raw_total"])
print("Remaining IDs:", 400 - scan["unique_total"])
print("Paid scraping: LOCKED")
print("=" * 60)


BATCH 7 — PART 12 OFFLINE VALIDATION
Status       : PASS
Saved tasks  : 59
RAW total    : 1379
Unique IDs   : 372
Remaining RAW: 121
Remaining IDs: 28
Paid scraping: LOCKED


In [30]:
# 12 — CONTOH TWEET UNTUK AUDIT SEMANTIK (TANPA API)
def preview_policy_samples(max_each_case=5):
    if max_each_case < 1:
        raise ValueError('max_each_case minimal 1')
    state = load_checkpoint()
    for case in FULL_CASES:
        print('\n' + '=' * 70 + '\nCONTOH:', case)
        shown, seen = 0, set()
        for record in state.values():
            if record.get('status') != 'SAVED' or record.get('case') != case:
                continue
            rel = record.get('raw_path')
            if not rel:
                continue
            path = PROJECT_ROOT / Path(str(rel).replace('\\', '/'))
            if not path.exists():
                continue
            with path.open('r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    if item.get('type') != 'tweet' or not item.get('id'):
                        continue
                    key = str(item['id'])
                    if key in seen:
                        continue
                    seen.add(key)
                    print('-', str(item.get('text', ''))[:350].replace('\n', ' '))
                    shown += 1
                    if shown >= max_each_case:
                        break
            if shown >= max_each_case:
                break
        if not shown:
            print('(Tidak ada tweet tersimpan yang tersedia untuk contoh)')

# Preview ini bukan labeling dan bukan bukti representativitas.


## Keputusan produksi dan catatan metodologis

- Jangan menggunakan nilai `CREDIT_USED_TO_DATE_USD` sebagai saldo yang dapat dibelanjakan. **Saldo aktual dashboard** wajib diisikan pada `AVAILABLE_CREDIT_USD` sebelum menjalankan batch berbayar.
- **Tidak ada panggilan API pada mode preview.** Cell `09B` memakai fake Actor dan direktori sementara. Cost lookup historis default OFF.
- `MAX_CHARGE_PER_RUN_USD` (per run API) ditambah komitmen maksimum setiap run dan `MAX_NEW_RUNS` (per sesi) adalah proteksi berlapis; nilai batas dapat berbeda dari tagihan platform karena pembulatan, peristiwa billing lain, atau biaya akun lain.
- Jika `RUN_MODE='full'`: satu query `E01_P01` per kasus, setiap 6 jam, total **416 task awal**. Tidak otomatis menyapu seluruh 42 grup keyword. Ini fase 1; perlu evaluasi yield dan jangkauan sebelum menambah query lain.
- `POSSIBLE_TRUNCATION` hanya alarm karena `unique_real_ids >= MAX_ITEMS`; tidak membuktikan jumlah tweet yang hilang. Pengujian 3 jam sebelumnya justru menemukan lebih sedikit ID unik dari 6 jam.
- Satu `SAVED` artinya file disimpan, **bukan** data semantik valid atau representatif. Saat raw->clean, hilangkan mock, duplikasi ID, duplikasi konten, promosi/berita yang tidak relevan, dan nilai kecukupan temporal.
- Jangan menyebut persentase sentimen sebagai representasi seluruh masyarakat Indonesia; basisnya adalah post X yang tersedia dalam sampel query.
